# Grape Bunch Detection & Counting - Vineyard Deployment
### YOLO11 Object Detection - Domain-Gap-Aware Training Mix - SAHI Tiled Inference - Manual Ground-Truth Annotation

**Task:** detect and count grape bunches in real farm photos (single class: `bunch`).

**Data sources** (see `grape_bunch_dataset/DATASET_PROVENANCE.md` for full provenance - shared with this project's classical-ML sister notebook):
- **Zenodo "Grapevine Bunch Detection Dataset"** (~10,010 images, CC BY 4.0) - close-up, mostly single-bunch, green-dominant, blurry (median box covers ~39% of the frame). Sourced as generic "grape bunch" web images.
- **CERTH Grape Dataset** (~2,587 images, prefixed `CERTH_`, CC BY 4.0) - real in-vineyard iPhone photos, several bunches per frame, sharp, small boxes (median ~3% of the frame). This is the multi-bunch/small-box/real-vineyard domain signal this notebook rebalances training around.
- **Live farm photos** (43 images) - real camera photos from an actual farm. Unlabeled until Phase 3 of this notebook.

**Why this rewrite exists:** the previous training run - which oversampled 300 Embrapa WGISD images 6x to reach ~11% of the training epoch - scored MAE 0.14 on the online in-distribution test set but MAE 1.34 (24% exact-match) on its WGISD holdout, and was never numerically evaluated on real farm photos at all, only eyeballed. WGISD has since been dropped project-wide (CC BY-NC 4.0, non-commercial-only, incompatible with this project's licensing needs - see `grape_bunch_dataset/DATASET_PROVENANCE.md`); CERTH fills the same "many small bunches, real vineyard framing" role at roughly 20x the image count and under an unrestricted license, though it is not a perfect substitute - CERTH's photos trend toward single/small-cluster close-ups rather than WGISD's wide multi-bunch trellis rows, so Phase 1.5's domain-gap diagnosis carries that caveat forward explicitly rather than claiming parity. Phase 1's EDA shows why the mix matters at all: the Zenodo (online) dataset does not resemble the deployment domain (single large centered bunch vs. many small occluded bunches scattered across a wide frame). This notebook quantifies that domain gap, rebalances training around it, adds SAHI tiled inference for the high-resolution farm photos, and builds a small hand-labeled real-photo holdout so "does it work in the field" finally has a number attached to it.

**Notebook scope:**
- **Phases 0-2** only need the 43 real farm photos and run on any machine with the images available locally - use them to sanity-check preprocessing before spending GPU time.
- **Phases 3-8** need the full `grape_bunch_dataset/` pool plus a GPU runtime. Both are already available locally in this repo (Option B below), so a local CUDA GPU (e.g. under WSL) is sufficient - Google Colab is now an optional alternative, not a requirement. Phase 3.1 additionally needs a few minutes of manual clicking to build the labeled holdout.


# Phase 0 - Environment Setup

## 0.0 Configure Base Path & Confirm GPU

Run exactly ONE of the next two cells, matching where this notebook is executing - not both. Everything after
Phase 0 only depends on `ROOT`, `LIVE_DIR`, and `DATASET_DIR` being set correctly, so the rest of the notebook
does not care which option was used.


### Option A - Google Colab (mounts Drive)

Use this if `grape_bunch_dataset/` and/or the real farm photos (the local `LiveImages/` folder, uploaded as `data/raw/`) live in Google Drive instead of
sitting next to this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU -- check runtime type")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

ROOT = "/content/drive/MyDrive/GrapeBunchDetection"
LIVE_DIR = f"{ROOT}/data/raw"   # real farm photos -- upload the local LiveImages/ folder's contents here

import os
for p in [
    "grape_bunch_dataset/images", "grape_bunch_dataset/labels",
    "data/raw", "data/processed", "data/eval",
    "data/annotations/live_holdout/images", "data/annotations/live_holdout/labels",
    "runs", "configs",
]:
    os.makedirs(f"{ROOT}/{p}", exist_ok=True)
print("Workspace ready at:", ROOT)
print("NOTE: upload grape_bunch_dataset/ and the contents of LiveImages/ (as data/raw/) to the path above before running Phase 1 onward.")


### Option B - Local machine / WSL (no Drive mount)

Use this if the notebook, the `grape_bunch_dataset/` folder, and the `data/` folder of real farm photos all sit
next to each other locally - nothing gets mounted, everything is read straight off disk. This is the path used
to train on a local CUDA GPU (e.g. an NVIDIA card passed through to WSL) instead of a Colab-hosted one, and is
now the primary way to run every phase of this notebook end-to-end, not just Phases 0-2.


In [ ]:
# --- Option B: local machine / WSL -- no Google Drive mount ---
import torch, subprocess
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU -- CPU only")
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi not found -- running on CPU-only or non-NVIDIA hardware.")

ROOT = str(Path.cwd())                       # local workspace: this notebook's own folder
LIVE_DIR = str(Path.cwd().parent / "LiveImages")  # the 43 real farm photos: LiveImages/ at the repo root, one level above this notebook

import os
for p in ["data/processed", "data/eval", "data/annotations/live_holdout/images",
          "data/annotations/live_holdout/labels", "runs", "configs"]:
    os.makedirs(f"{ROOT}/{p}", exist_ok=True)

n_live = len([p for p in Path(LIVE_DIR).glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
print("Workspace ready at:", ROOT)
print(f"Live farm photos: {n_live} images at {LIVE_DIR}")
print(f"Dataset pool: {ROOT}/grape_bunch_dataset (Zenodo + CERTH, already merged locally -- see Phase 1.1)")


## 0.1 Install Packages

In [ ]:
# !pip install -q ultralytics sahi ipympl ipywidgets grad-cam
import ultralytics
ultralytics.checks()

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not running in Colab -- no custom widget manager needed locally

## 0.2 Smoke-Test the Interactive Widget Backend & SAHI

Fail fast here instead of mid-notebook: confirm the `ipympl` widget backend renders (needed for Phase 3's
manual annotation tool) and that SAHI's model wrapper imports cleanly (needed for Phase 5's tiled inference).

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(2, 2))
ax.plot([0, 1], [0, 1])
ax.set_title("ipympl smoke test")
plt.show()
print("If an interactive plot rendered above (not a static image), the widget backend is working.")

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
print("SAHI import OK:", AutoDetectionModel, get_sliced_prediction)

## 0.3 Global Configuration

Seeds, class list, path constants, and the data-mix / tiling constants used throughout the notebook. Changing a
value here (e.g. `TARGET_CERTH_SHARE`) is the single edit point for re-running downstream phases with a
different configuration. `TRAIN_RATIO`/`VAL_RATIO`/`TEST_RATIO` match this project's classical-ML sister
notebook (60/20/20, stratified by source) so the two notebooks' splits are directly comparable.


In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASS_NAMES = ["bunch"]
NUM_CLASSES = len(CLASS_NAMES)

import sys
IN_COLAB = "google.colab" in sys.modules

DATASET_DIR = f"{ROOT}/grape_bunch_dataset"      # merged Zenodo + CERTH pool (see Phase 1.1)
CERTH_PREFIX = "CERTH_"                          # marks the CERTH subset for stratified splitting (matches the ML sister notebook's constant)

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.6, 0.2, 0.2   # image-level split ratios, same as the ML sister notebook

LOCAL_DATA = "/content/data/processed" if IN_COLAB else f"{ROOT}/data/processed"   # /content is fast local disk on Colab, not Drive

# Training / tiling constants
TRAIN_IMGSZ = 1024          # native tile size for both training and SAHI slices
TILE_SIZE = 1024
TILE_OVERLAP = 0.25         # SAHI overlap_height/width_ratio
TARGET_CERTH_SHARE = 0.5    # target share of the effective training epoch that should be CERTH-domain

print("Config set. Classes:", CLASS_NAMES)
print("LIVE_DIR:", LIVE_DIR)


In [ ]:
import cv2
from pathlib import Path

def sharpness(img_path, resize_to=1024):
    """Laplacian-variance blur/sharpness score, scale-normalized so resolution alone doesn't confound the comparison."""
    im = cv2.imread(str(img_path))
    if im is None:
        return None
    h, w = im.shape[:2]
    scale = resize_to / max(h, w)
    im = cv2.resize(im, (int(w * scale), int(h * scale)))
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def box_area_frac(yolo_line):
    """Normalized box area (w*h) from a single YOLO-format label line."""
    _, cx, cy, w, h = map(float, yolo_line.split())
    return w * h

print("Helpers ready: sharpness(), box_area_frac()")

# Phase 1 - Dataset Preparation & EDA

## 1.1 Data Acquisition & Structure Check

Confirm the expected folder layout is present before doing any real work. `grape_bunch_dataset/` is a flat pool
(no pre-split train/val/test folders) merging two CC BY 4.0 sources - see `grape_bunch_dataset/DATASET_PROVENANCE.md`
for full provenance, and the classical-ML sister notebook's Section 1.1 for the same load logic applied there.


In [ ]:
from pathlib import Path
import numpy as np
from tqdm import tqdm

IMAGES_DIR = Path(DATASET_DIR, "images")
LABELS_DIR = Path(DATASET_DIR, "labels")

assert IMAGES_DIR.exists() and LABELS_DIR.exists(), (
    f"Dataset not found at {DATASET_DIR}. Expected an 'images/' and 'labels/' "
    f"subfolder (YOLO-format .txt per image, one 'class cx cy w h' line per bunch)."
)

def read_yolo_lines(label_path):
    """Read a YOLO-format label file, return raw (class_id, cx, cy, w, h) string tuples, all normalized."""
    if not label_path.exists():
        return []
    lines = []
    for line in open(label_path):
        parts = line.split()
        if len(parts) >= 5:
            lines.append(tuple(parts[:5]))
    return lines

image_paths = sorted([p for p in IMAGES_DIR.glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
print(f"Images found: {len(image_paths)}")

per_image_counts, class_ids_seen = [], set()
for img_path in tqdm(image_paths, desc="Scanning labels"):
    boxes = read_yolo_lines(LABELS_DIR / f"{img_path.stem}.txt")
    per_image_counts.append(len(boxes))
    class_ids_seen.update(b[0] for b in boxes)

per_image_counts = np.array(per_image_counts)
print(f"Total bunch boxes: {per_image_counts.sum()}")
print(f"Bunches/image - mean: {per_image_counts.mean():.2f}  median: {np.median(per_image_counts):.0f}  "
      f"min: {per_image_counts.min()}  max: {per_image_counts.max()}")
print(f"Raw class ids present in label files: {sorted(class_ids_seen)} -- remapped to a single class 0 in Section 1.3")

n_live = len([p for p in Path(LIVE_DIR).glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
print(f"\nLive farm photos: {n_live} files at {LIVE_DIR}")


## 1.2 Identify the CERTH Subset (Multi-Bunch, Small-Box Domain)

The CERTH Grape Dataset (`CERTH_`-prefixed filenames, already merged into `grape_bunch_dataset/` - see
`DATASET_PROVENANCE.md`) supplies the multi-bunch, small-box, real-vineyard domain that the Zenodo half of the
pool lacks. Unlike WGISD, which this replaced, no external clone is needed here - it already sits on local disk
under the same CC BY 4.0 license as the rest of the pool.


In [ ]:
certh_paths = [p for p in image_paths if p.stem.startswith(CERTH_PREFIX)]
zenodo_paths = [p for p in image_paths if not p.stem.startswith(CERTH_PREFIX)]
print(f"CERTH subset:  {len(certh_paths)} images ({len(certh_paths) / len(image_paths):.1%} of the pool)")
print(f"Zenodo subset: {len(zenodo_paths)} images ({len(zenodo_paths) / len(image_paths):.1%} of the pool)")


### Validate the CERTH Label Format Before Trusting It

The same sanity check the previous notebook skipped for WGISD, applied here to the new source before it drives
any training decisions: parse a handful of CERTH label files under the assumed normalized `class cx cy w h`
convention and overlay the boxes on the image - if they hug real bunches, the assumption is confirmed; if not,
this is caught here instead of silently corrupting training.


In [ ]:
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

def find_image(stem):
    matches = [p for p in IMAGES_DIR.glob(stem + ".*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    return matches[0]

def overlay_yolo_boxes(img_path, label_path, color="red"):
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    w, h = im.size
    n = 0
    for line in open(label_path):
        if not line.strip():
            continue
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
        x2, y2 = (cx + bw / 2) * w, (cy + bh / 2) * h
        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
        n += 1
    return im, n

certh_label_files = [LABELS_DIR / f"{p.stem}.txt" for p in certh_paths]
picks = random.sample(certh_label_files, 4)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, lf in zip(axes, picks):
    im, n = overlay_yolo_boxes(find_image(lf.stem), lf)
    ax.imshow(im); ax.axis("off"); ax.set_title(f"{lf.stem}\n{n} boxes")
plt.suptitle("CERTH label sanity check -- boxes should tightly hug real bunches")
plt.tight_layout(); plt.show()
print("If boxes above sit correctly on bunches, the normalized 'class cx cy w h' format is confirmed for training use.")


## 1.3 Image-Level Train/Val/Test Split & Single-Class Remap

Both sources encode multiple class ids in their label files (1.1 found more than just `0` present), but this
project only cares about detecting a bunch as a single class. Split at the image level first - stratified by
source (CERTH vs. Zenodo), same 60/20/20 ratio and seed as the ML sister notebook - then copy each split's
images to a working directory while rewriting every label line to class id `0` (`bunch`) in the same pass.
Splitting before any downstream cropping/mixing matters for the same reason the ML notebook flags: patches or
mixes built from the same photo would otherwise leak information between train and test, so the raw image list
is split first and every derived file inherits its source image's split.


In [ ]:
import shutil, time
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def _source(path_str):
    return "CERTH" if Path(path_str).stem.startswith(CERTH_PREFIX) else "Zenodo"

pool_df = pd.DataFrame({"image_path": [str(p) for p in image_paths]})
pool_df["source"] = pool_df["image_path"].map(_source)

train_df, temp_df = train_test_split(
    pool_df, test_size=(VAL_RATIO + TEST_RATIO), random_state=SEED, stratify=pool_df["source"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=SEED, stratify=temp_df["source"]
)
print("Train:", train_df["source"].value_counts().to_dict())
print("Val:  ", val_df["source"].value_counts().to_dict())
print("Test: ", test_df["source"].value_counts().to_dict())

def remap_split(df, split_name):
    """Copy pool images to a working directory while rewriting every label line to class 0."""
    dst_img, dst_lbl = Path(LOCAL_DATA, split_name, "images"), Path(LOCAL_DATA, split_name, "labels")
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    for image_path in tqdm(df["image_path"], desc=split_name):
        p = Path(image_path)
        shutil.copy(p, dst_img / p.name)
        lf = LABELS_DIR / f"{p.stem}.txt"
        lines_out = []
        if lf.exists():
            for line in open(lf):
                parts = line.split()
                if not parts:
                    continue
                _, cx, cy, w, h = parts[:5]
                lines_out.append(f"0 {cx} {cy} {w} {h}")
        with open(dst_lbl / f"{p.stem}.txt", "w") as f:
            f.write("\n".join(lines_out) + ("\n" if lines_out else ""))

t0 = time.time()
for df, name in [(train_df, "train"), (val_df, "val"), (test_df, "test")]:
    remap_split(df, name)
print(f"Split + remap done in {time.time() - t0:.1f}s")


In [ ]:
for split in ["train", "val", "test"]:
    img_c = len(os.listdir(Path(LOCAL_DATA, split, "images")))
    lbl_c = len(os.listdir(Path(LOCAL_DATA, split, "labels")))
    print(split, "images:", img_c, "labels:", lbl_c)

classes_found = set()
for split in ["train", "val", "test"]:
    for lf in Path(LOCAL_DATA, split, "labels").glob("*.txt"):
        for line in open(lf):
            if line.strip():
                classes_found.add(line.split()[0])
print("Classes found after remap:", classes_found)

total_check = len(train_df) + len(val_df) + len(test_df)
assert total_check == len(pool_df), "split sizes don't add up to the full pool"
print(f"Split sizes sum to the full pool ({total_check} images) -- no leakage possible by construction "
      f"(sklearn's train_test_split partitions disjointly).")


## 1.4 CERTH Train / Val / Holdout Roles

WGISD only shipped a `train.txt`/`test.txt` list, so the previous version had to manually carve a "dev" slice
out of its own train pool for checkpoint selection. The stratified split in 1.3 already gives CERTH proper
train/val/test slices like every other split here, so the roles fall out directly:

- **CERTH train slice** - the oversample-listing pool for Section 1.6/1.7's training mix.
- **CERTH val slice** - folded into the validation mix (Section 1.7) for checkpoint selection during training.
- **CERTH test slice** - the CERTH holdout, never trained or validated on, reserved for Phase 5's domain-gap evaluation.

No separate copy step or leak-check is needed beyond what 1.3 already guaranteed - the split happened once, at
the image level, before any of these roles were assigned.


In [ ]:
def _split_paths(df, split_name):
    img_dir, lbl_dir = Path(LOCAL_DATA, split_name, "images"), Path(LOCAL_DATA, split_name, "labels")
    certh_names = df[df["source"] == "CERTH"]["image_path"].map(lambda p: Path(p).name)
    zenodo_names = df[df["source"] == "Zenodo"]["image_path"].map(lambda p: Path(p).name)
    return {
        "certh_images": [img_dir / n for n in certh_names],
        "certh_labels": [lbl_dir / f"{Path(n).stem}.txt" for n in certh_names],
        "zenodo_images": [img_dir / n for n in zenodo_names],
        "zenodo_labels": [lbl_dir / f"{Path(n).stem}.txt" for n in zenodo_names],
    }

train_split = _split_paths(train_df, "train")
val_split = _split_paths(val_df, "val")
test_split = _split_paths(test_df, "test")

print(f"CERTH train pool: {len(train_split['certh_images'])} images (Section 1.6/1.7 oversample-listing pool)")
print(f"CERTH val:        {len(val_split['certh_images'])} images (folded into val_mix, Section 1.7)")
print(f"CERTH holdout:    {len(test_split['certh_images'])} images (untouched, Phase 5 domain-gap eval)")
print(f"Zenodo train pool: {len(train_split['zenodo_images'])} images")


## 1.5 Per-Source EDA -- Bunches/Image, Box Size, Sharpness

This is the evidence base for the domain-mismatch diagnosis this whole rewrite is built around.
Compute the same three statistics for every source: how many bunches appear per image, how much of
the frame a typical box covers, and how sharp (in-focus) the photo is.

In [ ]:
def source_stats(label_files, image_files, n_sharp=20):
    counts, areas = [], []
    for lf in label_files:
        lines = [l for l in open(lf) if l.strip()] if lf.exists() else []
        counts.append(len(lines))
        areas.extend(box_area_frac(l) for l in lines)
    sharp_sample = random.sample(image_files, min(n_sharp, len(image_files)))
    sharp = [s for s in (sharpness(p) for p in sharp_sample) if s is not None]
    return {"counts": np.array(counts), "areas": np.array(areas), "sharp": np.array(sharp), "n_images": len(image_files)}

online_stats = source_stats(train_split["zenodo_labels"], train_split["zenodo_images"])
certh_stats = source_stats(train_split["certh_labels"], train_split["certh_images"])

live_img_paths = [p for p in Path(LIVE_DIR).glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
live_sharp = np.array([s for s in (sharpness(p) for p in random.sample(live_img_paths, min(20, len(live_img_paths)))) if s is not None])

for name, s in [("Online (Zenodo) train", online_stats), ("CERTH train", certh_stats)]:
    c, a, sh = s["counts"], s["areas"], s["sharp"]
    print(f"--- {name} ({s['n_images']} images) ---")
    print(f"  bunches/image: mean={c.mean():.2f} median={np.median(c):.0f} min={c.min()} max={c.max()}")
    print(f"  box area frac: mean={a.mean():.4f} median={np.median(a):.4f}")
    print(f"  sharpness:     mean={sh.mean():.1f} (n={len(sh)})")
print(f"--- Live farm photos ({len(live_img_paths)} images, unlabeled) ---")
print(f"  sharpness:     mean={live_sharp.mean():.1f} (n={len(live_sharp)})")


### Visualize the Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(online_stats["counts"], bins=range(0, int(online_stats["counts"].max()) + 2), color="steelblue")
axes[0, 0].set_title("Bunches per image - online (Zenodo) dataset")
axes[0, 1].hist(certh_stats["counts"], bins=range(0, int(certh_stats["counts"].max()) + 2), color="darkorange")
axes[0, 1].set_title("Bunches per image - CERTH train pool")

axes[1, 0].hist(online_stats["areas"], bins=40, color="steelblue")
axes[1, 0].set_title("Box area fraction - online (Zenodo) dataset")
axes[1, 1].hist(certh_stats["areas"], bins=40, color="darkorange")
axes[1, 1].set_title("Box area fraction - CERTH train pool")

plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
plt.hist(online_stats["sharp"], bins=15, alpha=0.6, label="online (Zenodo)", color="steelblue")
plt.hist(certh_stats["sharp"], bins=15, alpha=0.6, label="CERTH", color="darkorange")
plt.hist(live_sharp, bins=15, alpha=0.6, label="live farm photos", color="seagreen")
plt.legend(); plt.title("Sharpness (Laplacian variance) by source"); plt.xlabel("sharpness score")
plt.show()


### Sample Images Across Sources -- What Does Each Domain Actually Look Like?

In [ ]:
def sample_and_draw(label_files, image_files, n=3):
    picks = random.sample(list(zip(image_files, label_files)), n)
    return [overlay_yolo_boxes(img_path, lf)[0] for img_path, lf in picks]

online_samples = sample_and_draw(train_split["zenodo_labels"], train_split["zenodo_images"])
certh_samples = sample_and_draw(train_split["certh_labels"], train_split["certh_images"])
live_samples = [Image.open(p).convert("RGB") for p in random.sample(live_img_paths, 3)]

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for i, im in enumerate(online_samples):
    axes[0, i].imshow(im); axes[0, i].axis("off")
for i, im in enumerate(certh_samples):
    axes[1, i].imshow(im); axes[1, i].axis("off")
for i, im in enumerate(live_samples):
    axes[2, i].imshow(im); axes[2, i].axis("off")
axes[0, 1].set_title("ONLINE (Zenodo) - single large centered bunch, ground-truth boxes drawn", fontsize=11)
axes[1, 1].set_title("CERTH - several small bunches, ground-truth boxes drawn", fontsize=11)
axes[2, 1].set_title("LIVE FARM PHOTOS - many small occluded bunches, no labels yet", fontsize=11)
plt.tight_layout(); plt.show()


### Diagnosis: Why the Previous Run Failed on Real Images

The numbers above (and the sample grid) tell the same story from three independent angles - now measured
directly against this project's actual local pool rather than assumed:

| | Zenodo (online) | CERTH | Live farm photos |
|---|---|---|---|
| Bunches/image | ~1.2 (near-single centered subject) | ~3.8, small | many, small (visual match to CERTH) |
| Box size | huge (median ~39% of the frame) | small (median ~3% of the frame) | small (visual match to CERTH) |
| Sharpness | low (blurry, product-photo style, mean ~114) | high (mean ~936) | highest (mean ~3407) |

Zenodo - roughly 80% of the merged pool - teaches the model to expect "one big centered blob." Real deployment
images look much closer to CERTH: multiple small, leaf-occluded bunches scattered across a wide frame. The
previous WGISD-based run reached this same conclusion with an even starker contrast (WGISD's photos ran closer
to a handful of tiny boxes scattered across a *wide trellis-row* frame). CERTH is not a full substitute for that
framing: its photos trend toward single/small-cluster close-ups rather than WGISD's wide multi-bunch trellis
rows, so this rewrite's real-photo domain gap may still be partially open on the framing/occlusion axis even
though the bunch-count and box-size axes are now addressed by the swap (see Phase 8.4 for a follow-up
recommendation on this specific gap).

The rest of this notebook rebalances training toward the CERTH/live domain (Section 1.6 below), without
discarding Zenodo entirely - it still teaches generic "what does a grape bunch look like" appearance that is
worth keeping as a minority signal.


## 1.6 Data-Mix Formula - Downweight Zenodo, Weight Toward CERTH

Same target as the previous WGISD-based version: **CERTH-domain images should make up about 50% of the
effective training epoch**. The difference is scale - CERTH's train-split pool is roughly 20x larger than
WGISD's old 121-image pool, so hitting the same 50% target no longer requires heavily repeating the minority
source; it is reached mostly (or entirely) by subsampling Zenodo down to match, computed live below rather than
assumed up front:

```
share = (k * pool_size) / (k * pool_size + S_o)   =>   S_o = k * pool_size * (1/share - 1)
```

`k` (the CERTH repeat factor) is chosen as the smallest integer that keeps `S_o` from exceeding the available
Zenodo train pool - at CERTH's actual measured pool size this works out to `k=1` (no repetition needed at all),
unlike WGISD's `k=8`.


In [ ]:
certh_pool_size = len(train_split["certh_images"])
zenodo_pool_size = len(train_split["zenodo_images"])

k = 1
while True:
    S_o = round(k * certh_pool_size * (1 / TARGET_CERTH_SHARE - 1))
    if S_o <= zenodo_pool_size or k >= 8:
        break
    k += 1
CERTH_OVERSAMPLE = k

print(f"CERTH train pool: {certh_pool_size} images  x{CERTH_OVERSAMPLE} oversample = {certh_pool_size * CERTH_OVERSAMPLE} effective listings")
print(f"Zenodo subsample target: S_o = {S_o} ({S_o / zenodo_pool_size:.1%} of the {zenodo_pool_size} available Zenodo train images)")
print(f"Realized CERTH share: {(certh_pool_size * CERTH_OVERSAMPLE) / (certh_pool_size * CERTH_OVERSAMPLE + S_o):.1%} (target was {TARGET_CERTH_SHARE:.0%})")


### Online Subsampling: Weighted Toward the CERTH Domain, Not Uniform-Random

Instead of a hard random cut, rank Zenodo train images by how close they already are to CERTH's
(box-area-fraction, sharpness) centroid, and sample with probability inversely proportional to that distance.
This softly biases the subsample toward the Zenodo images that already look more like the target domain
(already sharper, already smaller-box), without hard-excluding the rest of the diversity Zenodo offers.


In [ ]:
def image_stats(lf, img_path):
    lines = [l for l in open(lf) if l.strip()] if lf.exists() else []
    area = np.mean([box_area_frac(l) for l in lines]) if lines else 0.0
    sh = sharpness(img_path)
    return area, sh

certh_area_mean, certh_sharp_mean = certh_stats["areas"].mean(), certh_stats["sharp"].mean()
certh_area_std, certh_sharp_std = certh_stats["areas"].std() + 1e-6, certh_stats["sharp"].std() + 1e-6

rows = []
for img_path, lf in tqdm(list(zip(train_split["zenodo_images"], train_split["zenodo_labels"])), desc="scoring Zenodo images"):
    area, sh = image_stats(lf, img_path)
    if sh is None:
        continue
    z_area = (area - certh_area_mean) / certh_area_std
    z_sharp = (sh - certh_sharp_mean) / certh_sharp_std
    dist = np.sqrt(z_area ** 2 + z_sharp ** 2)
    rows.append((img_path, lf, dist))

scored_images, scored_labels, dists = zip(*rows)
dists = np.array(dists)
weights = 1.0 / (dists + 1.0)          # +1 avoids division blowups for near-zero distance
weights = weights / weights.sum()

rng = np.random.default_rng(SEED)
chosen_idx = rng.choice(len(scored_images), size=min(S_o, len(scored_images)), replace=False, p=weights)
zenodo_subsample_images = [scored_images[i] for i in chosen_idx]
zenodo_subsample_labels = [scored_labels[i] for i in chosen_idx]
print(f"Selected {len(zenodo_subsample_images)} Zenodo images via inverse-distance-weighted sampling.")


## 1.7 Materialize the Mixed Training File List & data.yaml

In [ ]:
certh_train_images = train_split["certh_images"]

train_mix = zenodo_subsample_images + certh_train_images * CERTH_OVERSAMPLE
random.Random(SEED).shuffle(train_mix)

train_mix_path = Path(LOCAL_DATA, "train_mix.txt")
with open(train_mix_path, "w") as f:
    f.writelines(str(p.resolve()) + "\n" for p in train_mix)

print(f"Zenodo: {len(zenodo_subsample_images)} | CERTH: {len(certh_train_images)} x{CERTH_OVERSAMPLE} = {len(certh_train_images) * CERTH_OVERSAMPLE} | Total: {len(train_mix)}")
print(f"Realized CERTH share of epoch: {(len(certh_train_images) * CERTH_OVERSAMPLE) / len(train_mix):.1%}")


### Assemble the Validation Set

Validation should reflect the deployment target too, not just the Zenodo domain: mix a modest Zenodo-val
subsample with CERTH's own val slice from Section 1.4 (no separate manual "dev" carve-out needed now that CERTH
has a real val split of its own, unlike WGISD).


In [ ]:
online_val_subsample = random.Random(SEED).sample(val_split["zenodo_images"], min(400, len(val_split["zenodo_images"])))
certh_val_images = val_split["certh_images"]

val_mix = online_val_subsample + certh_val_images
val_mix_path = Path(LOCAL_DATA, "val_mix.txt")
with open(val_mix_path, "w") as f:
    f.writelines(str(p.resolve()) + "\n" for p in val_mix)

print(f"Val mix: {len(online_val_subsample)} Zenodo + {len(certh_val_images)} CERTH = {len(val_mix)} images")


In [ ]:
yaml_content = f"""path: {LOCAL_DATA}
train: train_mix.txt
val: val_mix.txt
test: test/images
nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""

with open(Path(LOCAL_DATA, "data.yaml"), "w") as f:
    f.write(yaml_content)

os.makedirs(f"{ROOT}/configs", exist_ok=True)
with open(f"{ROOT}/configs/data.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

# Phase 2 - Image Analysis Pipeline

This phase runs directly against the 43 real farm photos, since they are the only data available
outside Colab/Drive -- and they are the domain that actually matters for deployment.

**A note on what this phase is (and is not), unlike the classical-ML sister notebook:** in a
handcrafted-feature pipeline (CLAHE -> Gabor -> LBP -> HOG -> ... -> feature vector -> SVM/XGBoost),
every technique here would be manually vectorized and fed into the model. A YOLO detector's CNN
backbone learns its own features directly from (lightly preprocessed) pixels -- nothing below gets
manually concatenated into a feature vector. Instead, each technique here is a **diagnostic and
augmentation/tiling-justification tool**: it explains *why* the training configuration in Phase 3
looks the way it does, and gives a quick visual sanity check before spending GPU time.

## 2.1 Sharpness / Blur Quantification

### a) Intuition
The `sharpness()` helper (Phase 0) scores focus quality via the variance of the Laplacian: a sharp,
detailed image has high-frequency edges everywhere, so the second-derivative filter response varies a
lot; a blurry image has smoothed-out edges, so the response is nearly constant (low variance). This is
already the single strongest domain signal seen in Phase 1: live photos average ~3145, WGISD ~1465,
online ~97 -- more than a 30x gap between the training-dominant source and the deployment target.

### b) Implementation
Reused directly from Phase 0. Below: sharpness spread across the local live photos, which will later
be used to pick a representative annotation subset in Phase 3.

In [ ]:
local_live_paths = sorted([p for p in Path(LIVE_DIR).glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
local_live_sharp = {p: sharpness(p) for p in local_live_paths}

vals = np.array(list(local_live_sharp.values()))
print(f"Local live photos: n={len(vals)}  mean={vals.mean():.1f}  min={vals.min():.1f}  max={vals.max():.1f}")

plt.figure(figsize=(8, 4))
plt.hist(vals, bins=15, color="seagreen")
plt.title("Sharpness distribution -- local real farm photos")
plt.xlabel("Laplacian-variance sharpness"); plt.ylabel("count")
plt.show()


## 2.2 CLAHE -- Contrast Limited Adaptive Histogram Equalization

### a) Intuition
Farm photos are shot under dappled outdoor light: direct sun on some leaves, deep shadow under the
canopy where many bunches hang, in the same frame. Global contrast stretching would blow out the sunlit
regions to recover shadow detail (or vice versa). CLAHE instead equalizes contrast *locally*, in small
tiles, and clips the histogram so noise in near-flat regions doesn't get amplified -- exactly the
outdoor, uneven-lighting scenario these photos present.

### b) Implementation
Apply CLAHE on the L (luminance) channel only, in LAB color space, so color balance is preserved.

In [ ]:
def apply_clahe(img_bgr, clip_limit=2.0, tile_size=(8, 8)):
    l, a, b = cv2.split(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    l_eq = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)

demo_paths = random.sample(local_live_paths, 3)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, p in enumerate(demo_paths):
    img = cv2.imread(str(p))
    clahe_img = apply_clahe(img)
    axes[0, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0, i].axis("off")
    axes[1, i].imshow(cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB)); axes[1, i].axis("off")
axes[0, 0].set_ylabel("original"); axes[1, 0].set_ylabel("CLAHE")
plt.suptitle("Top: original | Bottom: CLAHE-enhanced -- shadowed bunches under canopy should become more visible")
plt.tight_layout(); plt.show()

## 2.3 Edge-Preserving Denoising

### a) Intuition
At native resolution these photos are sharp, but outdoor JPEGs from a phone/camera still carry sensor
noise and compression artifacts that can look like spurious high-frequency edges -- exactly the kind of
signal a small-object detector is sensitive to (a real bunch and a noise cluster are not that
different in scale). Unlike 2.1's Laplacian-variance sharpness score (which responds to *all*
high-frequency content, wanted or not), the noise estimate below uses a kernel specifically constructed
to null out on real edges/gradients and respond only to the noise-like residual, so it can separate
"blurry" (2.1's concern) from "noisy" (this section's concern) instead of conflating the two. A bilateral
filter smooths flat regions while explicitly preserving edges, so it reduces noise without blurring the
bunch/leaf boundaries the detector actually needs -- quantified below rather than judged by eye alone.

### b) Implementation
Fast noise-sigma estimate (Immerkaer, 1996): convolve with a 3x3 kernel that sums to zero on any locally
planar or linear gradient (so real edges contribute ~0) but responds strongly to independent per-pixel
noise, then normalize by the kernel's known scaling factor to recover an estimated noise standard
deviation in the same 0-255 intensity units as the image.


**Role of this section, honestly stated:** unlike 2.1/2.2/2.4/2.5, this diagnostic does not currently feed
a specific downstream config value (no augmentation parameter or pipeline setting is derived from the noise
number below) -- it is kept as a completeness check ("is noise actually a problem here") rather than a
justification for a design choice. The answer it gives is no: farm photos are already sharp (2.1's mean
~3400) and the noise reduction here is modest (~37%), so denoising is not the leverage point -- the
recall-failure diagnosis in Phase 6.3b/6.5 points at training-data density (Phase 8.4), not image noise.

In [ ]:
def estimate_noise_sigma(img_bgr):
    """Immerkaer (1996) fast noise-sigma estimate -- see 2.3(b) for what the kernel does and why."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float64)
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float64)
    conv = cv2.filter2D(gray, -1, kernel, borderType=cv2.BORDER_REFLECT)[1:-1, 1:-1]  # drop the 1px kernel border
    return np.sqrt(np.pi / 2) * np.mean(np.abs(conv)) / 6

def denoise(img_bgr, d=9, sigma_color=75, sigma_space=75):
    return cv2.bilateralFilter(img_bgr, d, sigma_color, sigma_space)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sigmas_before, sigmas_after = [], []
for i, p in enumerate(demo_paths):
    img = cv2.imread(str(p))
    den = denoise(img)
    sb, sa = estimate_noise_sigma(img), estimate_noise_sigma(den)
    sigmas_before.append(sb); sigmas_after.append(sa)
    axes[0, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0, i].axis("off")
    axes[0, i].set_title(f"noise sigma={sb:.2f}", fontsize=9)
    axes[1, i].imshow(cv2.cvtColor(den, cv2.COLOR_BGR2RGB)); axes[1, i].axis("off")
    axes[1, i].set_title(f"noise sigma={sa:.2f}  (-{(1 - sa / sb) * 100:.0f}%)", fontsize=9)
plt.suptitle("Top: original | Bottom: bilateral-filtered -- flat regions smoothed, edges preserved")
plt.tight_layout(); plt.show()

mean_reduction = (1 - np.mean(sigmas_after) / np.mean(sigmas_before)) * 100
print(f"Mean noise sigma: {np.mean(sigmas_before):.2f} -> {np.mean(sigmas_after):.2f}  ({mean_reduction:.0f}% reduction)")


## 2.4 HSV Color-Distribution Analysis

### a) Intuition
Grape bunches (dark purple/black in this variety) and leaf canopy (green) sit in different parts of hue
space, but overlap heavily in low-light shadow -- exactly where CLAHE (2.2) helps most. Quantifying how
separable bunch-hue is from leaf-hue tells us how aggressively the training augmentation can jitter hue
without destroying the very signal the detector needs (too much `hsv_h` jitter could push bunch color
into leaf-color territory and back, which would actively hurt rather than help).

### b) Implementation
Sample pixels from a manually-eyeballed dark-cluster region vs. the rest of the frame is overkill for a
notebook step -- instead, look at the overall hue histogram of a few images, which shows a clear green
peak (leaves, dominant) and a smaller dark/low-saturation cluster (bunches, fruit, shadow).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, p in enumerate(demo_paths):
    img = cv2.imread(str(p))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    axes[i].hist(hsv[:, :, 0].ravel(), bins=180, range=(0, 180), color="purple", alpha=0.7)
    axes[i].set_title(f"Hue histogram - {p.name[:12]}...")
    axes[i].set_xlabel("hue (0-180, OpenCV convention)")
plt.tight_layout(); plt.show()
print("A dominant green peak (~35-85) confirms leaf-canopy dominance; low-saturation/dark pixels (bunches, shadow)")
print("do not show as a clean hue peak since hue is undefined for near-black pixels -- consistent with the visual samples.")

## 2.5 Tiling Scheme Visualization

### a) Intuition
Real farm photos are ~2048x1536, but YOLO downsamples to its training `imgsz` (1024 in this project,
640 in the previous run) before the backbone ever sees them. A bunch that is, say, 120px wide in the
original photo shrinks to ~60px at 1024 or ~37px at 640 -- and many bunches in these photos are already
smaller than that before any downsampling, per the visual samples in Phase 1. Splitting the image into
overlapping tiles and running detection per-tile (SAHI, wired up in Phase 5) lets each tile be fed at
much closer to its native resolution, recovering the small-object detail a single full-frame pass would
lose.

### b) Implementation
Visualize the chosen tile grid (`TILE_SIZE=1024`, `TILE_OVERLAP=0.25`) over a real photo, and compare
an in-place tile crop against the same region after full-frame downsampling to 1024 -- the tile crop
should visibly preserve more bunch detail.

In [ ]:
def tile_grid(img_w, img_h, tile_size, overlap):
    step = int(tile_size * (1 - overlap))
    xs = list(range(0, max(img_w - tile_size, 0) + 1, step)) or [0]
    ys = list(range(0, max(img_h - tile_size, 0) + 1, step)) or [0]
    if xs[-1] + tile_size < img_w:
        xs.append(img_w - tile_size)
    if ys[-1] + tile_size < img_h:
        ys.append(img_h - tile_size)
    return [(x, y, min(x + tile_size, img_w), min(y + tile_size, img_h)) for y in ys for x in xs]

demo_img = cv2.imread(str(demo_paths[0]))
h, w = demo_img.shape[:2]
tiles = tile_grid(w, h, TILE_SIZE, TILE_OVERLAP)
print(f"Image {w}x{h} -> {len(tiles)} tiles of {TILE_SIZE}px with {TILE_OVERLAP:.0%} overlap")

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.imshow(cv2.cvtColor(demo_img, cv2.COLOR_BGR2RGB))
for (x1, y1, x2, y2) in tiles:
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor="red", facecolor="none", linewidth=2))
ax.set_title(f"{len(tiles)}-tile SAHI grid overlay"); ax.axis("off")
plt.show()

In [ ]:
x1, y1, x2, y2 = tiles[0]
native_tile = demo_img[y1:y2, x1:x2]
full_frame_downsized = cv2.resize(demo_img, (TRAIN_IMGSZ, int(h * TRAIN_IMGSZ / w)))
equivalent_region_h = int((y2 - y1) * TRAIN_IMGSZ / w)
equivalent_region_w = int((x2 - x1) * TRAIN_IMGSZ / w)
downsized_crop = full_frame_downsized[y1 * TRAIN_IMGSZ // w: y1 * TRAIN_IMGSZ // w + equivalent_region_h,
                                       x1 * TRAIN_IMGSZ // w: x1 * TRAIN_IMGSZ // w + equivalent_region_w]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(native_tile, cv2.COLOR_BGR2RGB)); axes[0].set_title("Native-resolution tile crop"); axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(downsized_crop, cv2.COLOR_BGR2RGB)); axes[1].set_title("Same region after full-frame downsize"); axes[1].axis("off")
plt.suptitle("Same region: tiled (left) preserves far more bunch/leaf detail than a single full-frame pass (right)")
plt.tight_layout(); plt.show()

# Phase 3 - Dataset Assembly & Annotation

## 3.1 In-Notebook Manual Bounding-Box Annotation Tool

None of the 43 live farm photos have ground-truth labels, so real-world performance has never actually
been measured -- only eyeballed. This section builds a small labeled holdout by hand-annotating a
diversity-picked subset directly inside the notebook (no external app needed), so Phase 5 can report a
real MAE/precision/recall on the deployment domain instead of only proxy-domain metrics.

**Interaction model:** two clicks per box, not click-and-drag -- click 1 sets the top-left corner (a
dashed preview follows the mouse), click 2 sets the bottom-right corner and commits a solid box.
Buttons drive navigation/undo/redo/save (not keyboard shortcuts), because Colab's websocket-relayed
canvas can drop continuous drag-event sequences, but discrete button clicks are reliable.

**Annotation guideline:** draw one box per visually distinct bunch cluster that is at least ~1/3
visible; if two bunches are touching and not clearly separable, box them together rather than guessing
a split. This is a single-annotator tool with no inter-annotator agreement check -- a stated limitation,
revisited in Phase 8.

Every add/undo/redo/navigate autosaves straight to a YOLO-format `.txt` (never only at the end), and
re-running this cell resumes any previously saved boxes -- so annotation can be done across multiple
sessions.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display
from PIL import Image


class BBoxAnnotator:
    """Two-click-corner bounding box annotator for a folder of images, saving YOLO-format labels."""

    def __init__(self, image_paths, label_dir, class_id=0):
        self.image_paths = list(image_paths)
        self.label_dir = Path(label_dir)
        self.label_dir.mkdir(parents=True, exist_ok=True)
        self.class_id = class_id
        self.idx = 0
        self.boxes = {}
        self.redo_stack = {}
        self._pending_corner = None
        self._preview_patch = None
        self._img_w = self._img_h = None

        self.out = widgets.Output()
        self.status = widgets.Label()
        self.dropdown = widgets.Dropdown(options=[p.stem for p in self.image_paths], description="Image:")
        self.btn_prev = widgets.Button(description="< Prev")
        self.btn_next = widgets.Button(description="Next >")
        self.btn_undo = widgets.Button(description="Undo")
        self.btn_redo = widgets.Button(description="Redo")
        self.btn_clear = widgets.Button(description="Clear image")
        self.btn_save = widgets.Button(description="Save", button_style="success")

        self.dropdown.observe(self._on_dropdown, names="value")
        self.btn_prev.on_click(lambda _: self._nav(-1))
        self.btn_next.on_click(lambda _: self._nav(1))
        self.btn_undo.on_click(lambda _: self._undo())
        self.btn_redo.on_click(lambda _: self._redo())
        self.btn_clear.on_click(lambda _: self._clear())
        self.btn_save.on_click(lambda _: self._save_current())

        controls = widgets.HBox([self.btn_prev, self.btn_next, self.btn_undo, self.btn_redo, self.btn_clear, self.btn_save])
        display(widgets.VBox([self.dropdown, controls, self.status, self.out]))

        self.fig, self.ax = plt.subplots(figsize=(10, 7.5))
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)
        self.fig.canvas.mpl_connect("motion_notify_event", self._on_move)
        self._load_image(0)

    def _stem(self):
        return self.image_paths[self.idx].stem

    def _label_path(self, stem):
        return self.label_dir / f"{stem}.txt"

    def _load_image(self, idx):
        self.idx = idx % len(self.image_paths)
        stem = self._stem()

        self.dropdown.unobserve(self._on_dropdown, names="value")
        self.dropdown.value = stem
        self.dropdown.observe(self._on_dropdown, names="value")

        img = Image.open(self.image_paths[self.idx]).convert("RGB")
        self._img_w, self._img_h = img.size

        if stem not in self.boxes:
            self.boxes[stem] = self._load_existing(stem)
            self.redo_stack.setdefault(stem, [])

        self._pending_corner = None
        self._preview_patch = None
        self.ax.clear()
        self.ax.imshow(np.array(img))
        self.ax.axis("off")
        self._redraw_boxes()
        with self.out:
            self.out.clear_output(wait=True)
            display(self.fig)
        self._update_status()

    def _load_existing(self, stem):
        lp = self._label_path(stem)
        if not lp.exists():
            return []
        boxes = []
        for line in open(lp):
            if not line.strip():
                continue
            _, cx, cy, w, h = map(float, line.split())
            x1 = (cx - w / 2) * self._img_w
            y1 = (cy - h / 2) * self._img_h
            x2 = (cx + w / 2) * self._img_w
            y2 = (cy + h / 2) * self._img_h
            boxes.append((x1, y1, x2, y2))
        return boxes

    def _redraw_boxes(self):
        for p in list(self.ax.patches):
            p.remove()
        for (x1, y1, x2, y2) in self.boxes[self._stem()]:
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none")
            self.ax.add_patch(rect)
        self.fig.canvas.draw_idle()

    def _update_status(self):
        n = len(self.boxes[self._stem()])
        self.status.value = f"{self.idx + 1}/{len(self.image_paths)}  {self._stem()}  --  {n} box(es)"

    def _on_click(self, event):
        if event.xdata is None or event.ydata is None or event.inaxes != self.ax:
            return
        x = min(max(event.xdata, 0), self._img_w)
        y = min(max(event.ydata, 0), self._img_h)
        if self._pending_corner is None:
            self._pending_corner = (x, y)
        else:
            x1, y1 = self._pending_corner
            box = (min(x1, x), min(y1, y), max(x1, x), max(y1, y))
            if box[2] - box[0] > 2 and box[3] - box[1] > 2:
                self.boxes[self._stem()].append(box)
                self.redo_stack[self._stem()] = []
                self._autosave()
            self._pending_corner = None
            if self._preview_patch is not None:
                self._preview_patch.remove()
                self._preview_patch = None
            self._redraw_boxes()
            self._update_status()

    def _on_move(self, event):
        if self._pending_corner is None or event.xdata is None or event.inaxes != self.ax:
            return
        x1, y1 = self._pending_corner
        if self._preview_patch is not None:
            self._preview_patch.remove()
        self._preview_patch = patches.Rectangle(
            (min(x1, event.xdata), min(y1, event.ydata)),
            abs(event.xdata - x1), abs(event.ydata - y1),
            linewidth=1.5, edgecolor="yellow", facecolor="none", linestyle="--",
        )
        self.ax.add_patch(self._preview_patch)
        self.fig.canvas.draw_idle()

    def _nav(self, step):
        self._load_image(self.idx + step)

    def _on_dropdown(self, change):
        stem = change["new"]
        self._load_image([p.stem for p in self.image_paths].index(stem))

    def _undo(self):
        stem = self._stem()
        if self.boxes[stem]:
            self.redo_stack[stem].append(self.boxes[stem].pop())
            self._autosave()
            self._redraw_boxes()
            self._update_status()

    def _redo(self):
        stem = self._stem()
        if self.redo_stack[stem]:
            self.boxes[stem].append(self.redo_stack[stem].pop())
            self._autosave()
            self._redraw_boxes()
            self._update_status()

    def _clear(self):
        stem = self._stem()
        self.redo_stack[stem] = list(reversed(self.boxes[stem]))
        self.boxes[stem] = []
        self._autosave()
        self._redraw_boxes()
        self._update_status()

    def _save_current(self):
        self._autosave()
        self.status.value = f"Saved {self._stem()}.txt ({len(self.boxes[self._stem()])} boxes)"

    def _autosave(self):
        stem = self._stem()
        lines = []
        for (x1, y1, x2, y2) in self.boxes[stem]:
            cx, cy = ((x1 + x2) / 2) / self._img_w, ((y1 + y2) / 2) / self._img_h
            bw, bh = (x2 - x1) / self._img_w, (y2 - y1) / self._img_h
            lines.append(f"{self.class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        # zero-bunch images get an explicit empty file -- valid YOLO background-image convention
        with open(self._label_path(stem), "w") as f:
            f.write("\n".join(lines))


print("BBoxAnnotator class ready.")
print("Fallback if ipympl misbehaves in your Colab session: display the image with a pixel-coordinate")
print("grid overlay and type box corners into an ipywidgets.Text field, then append directly to")
print("annotator.boxes[stem] and call annotator._autosave() -- same output format, just a slower input path.")

### Pick a Diversity-Spread Annotation Subset

Rather than the first N images alphabetically, spread the subset across the sharpness range already
computed in Phase 2 (2.1), so the holdout captures lighting/blur variety instead of a narrow slice of it.

In [ ]:
N_ANNOTATE = 43   # bumped from 24 to cover all live farm photos -- Phase 9's fine-tune needs >=35 labeled

live_paths_all = sorted([p for p in Path(LIVE_DIR).glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
live_sharp_all = {p.stem: sharpness(p) for p in live_paths_all}
sorted_by_sharp = sorted(live_sharp_all.items(), key=lambda kv: kv[1])

idxs = np.linspace(0, len(sorted_by_sharp) - 1, min(N_ANNOTATE, len(sorted_by_sharp))).round().astype(int)
annotate_stems = {sorted_by_sharp[i][0] for i in idxs}
annotate_paths = [p for p in live_paths_all if p.stem in annotate_stems]
print(f"Selected {len(annotate_paths)} of {len(live_paths_all)} live photos for manual annotation (sharpness-spread sample).")


### Run the Annotator

Work through as many of the selected images as you can (aim for all 43, but even a smaller subset still
produces a usable holdout -- Phase 8 reports statistical caveats for small N). Progress autosaves, so
this cell can be re-run across multiple sessions to resume.

In [ ]:
LIVE_HOLDOUT_LABELS = Path(ROOT, 'data/annotations/live_holdout/labels')

# Instantiates the widget UI itself -- BBoxAnnotator (3.1's class cell above) was previously only ever
# defined, never actually created, so running this section produced no visible tool. Re-running this cell
# is safe and resumes progress: __init__ loads any existing .txt labels per image (_load_existing) before
# showing the first one, so nothing already annotated is lost or overwritten.
annotator = BBoxAnnotator(annotate_paths, LIVE_HOLDOUT_LABELS)

### Finalize the Labeled Holdout (run once annotation is done)

In [ ]:
labeled = list(LIVE_HOLDOUT_LABELS.glob("*.txt"))
print(f"{len(labeled)} / {len(annotate_paths)} images annotated so far.")

LIVE_HOLDOUT_IMAGES = Path(ROOT, "data/annotations/live_holdout/images")
LIVE_HOLDOUT_IMAGES.mkdir(parents=True, exist_ok=True)
for p in annotate_paths:
    dst = LIVE_HOLDOUT_IMAGES / p.name
    if not dst.exists():
        shutil.copy(p, dst)

total_gt_bunches = sum(len([l for l in open(lf) if l.strip()]) for lf in labeled)
print(f"live_holdout ready at {LIVE_HOLDOUT_IMAGES.parent} -- {len(labeled)} labeled images, {total_gt_bunches} total ground-truth bunches")

## 3.2 Augmentation Configuration for Small, Occluded Objects

Boosted beyond Ultralytics defaults, justified directly by Phase 2's findings:

| Setting | Value | Why |
|---|---|---|
| `hsv_h` | 0.03 | Mild hue jitter -- Section 2.4 showed bunch/leaf hue separation is real but not huge; too much jitter risks pushing bunch color into leaf-color territory |
| `hsv_s`, `hsv_v` | 0.7, 0.5 | Larger saturation/value jitter to cover the wide lighting range CLAHE (2.2) was built to address |
| `scale` | 0.9 | Wide zoom-jitter range, so single-bunch Zenodo images and multi-bunch CERTH images both get exposed to a range of apparent object sizes |
| `mosaic` | 1.0 | Composites 4 images into one -- approximates a tile-like, multi-object scene from single-bunch Zenodo source images |
| `copy_paste` | 0.3 | Pastes bunch instances across images -- directly increases small-object density per image, compensating for not pre-tiling training data (Phase 5 tiles only at inference) |
| `close_mosaic` | 15 | Disable mosaic for the last 15 epochs so the model fine-tunes on realistic (non-composited) image statistics before training ends |


In [ ]:
AUGMENT_CONFIG = dict(
    hsv_h=0.03, hsv_s=0.7, hsv_v=0.5,
    scale=0.9, mosaic=1.0, copy_paste=0.3, close_mosaic=15,
)
print(AUGMENT_CONFIG)

### Mosaic-Batch Sanity Check -- Confirm Labels Stay Aligned Post-Augmentation

In [ ]:
from ultralytics.data import build_yolo_dataset
from ultralytics.cfg import get_cfg
from ultralytics.utils import DEFAULT_CFG

cfg = get_cfg(DEFAULT_CFG, overrides={**AUGMENT_CONFIG, "imgsz": TRAIN_IMGSZ, "data": f"{LOCAL_DATA}/data.yaml"})
train_ds = build_yolo_dataset(cfg, str(Path(LOCAL_DATA, "train_mix.txt")), batch=4, data={"names": CLASS_NAMES, "nc": NUM_CLASSES}, mode="train")

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for i, ax in enumerate(axes.ravel()):
    sample = train_ds[i]
    img = sample["img"].numpy().transpose(1, 2, 0)
    ax.imshow(img); ax.axis("off")
    boxes = sample["bboxes"].numpy() if len(sample["bboxes"]) else np.empty((0, 4))
    h, w = img.shape[:2]
    for cx, cy, bw, bh in boxes:
        x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
        ax.add_patch(plt.Rectangle((x1, y1), bw * w, bh * h, edgecolor="lime", facecolor="none", linewidth=1.5))
plt.suptitle("Augmented training batch sample -- boxes should still tightly hug bunches after mosaic/scale/copy-paste")
plt.tight_layout(); plt.show()

# Phase 4 - Model Training

## 4.1 Baseline Run (Previous Notebook) -- Recorded for Comparison, WGISD Row Now Historical-Only

No retrain needed to know these numbers - they are the previous run's own reported output (single-class
`yolo11m`, WGISD oversampled 6x to ~11% of the epoch, `imgsz=640`). The WGISD-holdout row can no longer be
recomputed - WGISD is no longer part of this project (see the notebook intro and `DATASET_PROVENANCE.md`) - so
it stays as a historical reference point only. Phase 5 onward compares against the new CERTH holdout instead,
which has no baseline counterpart to diff against (see Phase 8.2/8.3 for how that is handled honestly).


In [ ]:
BASELINE_METRICS = {
    "online_test":              {"mae": 0.14, "rmse": 0.46, "exact_match": 88.4, "bias": -0.01, "n": 1980},
    "wgisd_holdout_historical": {"mae": 1.34, "rmse": 1.87, "exact_match": 24.1, "bias": -0.10, "n": 29},
}
print("Baseline (previous run -- single-class yolo11m, 6x WGISD oversample, imgsz=640):")
print(f"  online_test: MAE={BASELINE_METRICS['online_test']['mae']} RMSE={BASELINE_METRICS['online_test']['rmse']} "
      f"Exact%={BASELINE_METRICS['online_test']['exact_match']} Bias={BASELINE_METRICS['online_test']['bias']} (n={BASELINE_METRICS['online_test']['n']})")
print(f"  wgisd_holdout_historical (WGISD dropped from this project -- historical reference only, not recomputable): "
      f"MAE={BASELINE_METRICS['wgisd_holdout_historical']['mae']} RMSE={BASELINE_METRICS['wgisd_holdout_historical']['rmse']} "
      f"Exact%={BASELINE_METRICS['wgisd_holdout_historical']['exact_match']} Bias={BASELINE_METRICS['wgisd_holdout_historical']['bias']} (n={BASELINE_METRICS['wgisd_holdout_historical']['n']})")


## 4.2 Train the Domain-Aware Model

`imgsz=1024` (vs. the previous 640) narrows the train/deployment resolution gap discussed in Phase 2.5. The
mixed training epoch, sized in Section 1.6/1.7, is smaller than the Zenodo-only scheme's full pool, so epoch
count is raised and `patience`-based early stopping is relied on against the domain-mixed validation set from
1.7 rather than a fixed epoch budget. `device=0` targets CUDA device 0 (confirmed available in Phase 0, whether
that is a local WSL GPU or a Colab-hosted one); Ultralytics auto-enables AMP mixed precision whenever training
runs on GPU, so no separate flag is needed for that.

**On `workers` and the multiprocessing start method:** Python 3.14 changed the default `multiprocessing` start
method away from `fork` (confirmed by direct testing in this environment: PyTorch DataLoader workers under the
new default crash with `ConnectionResetError` during forkserver handshake once CUDA is already initialized in
the parent process). Forcing `fork` explicitly below restores multi-worker data loading -- confirmed ~4x faster
per epoch in this environment than falling back to `workers=0`. This only matters on Linux/WSL; Colab runtimes
are unaffected since they still ship Python <3.14.

**Already trained this before?** Skip this cell and run the **4.3 Save & Load Trained Model** section below
instead to restore a checkpoint from disk without spending another ~180 epochs of GPU time. Note that
Ultralytics does **not** overwrite a previous run by default: re-running this training cell with the same
`RUN_NAME` creates `domain_aware_v1_2`, `_3`, etc. rather than skipping or erroring, so a stray re-run costs
GPU time but never loses a prior result.


In [ ]:
import multiprocessing
if multiprocessing.get_start_method(allow_none=True) != "fork":
    multiprocessing.set_start_method("fork", force=True)  # see Phase 4.2 markdown -- Python 3.14's new default breaks DataLoader workers here

from ultralytics import YOLO

model = YOLO("yolo11m.pt")

if IN_COLAB:
    # /content is fast local disk; Drive access is slow enough that syncing checkpoints back only at
    # epoch boundaries (not on every internal write) avoids stalling training on Drive I/O.
    def sync_to_drive(trainer):
        local_dir = Path(trainer.save_dir)
        drive_dir = Path(ROOT) / "runs" / local_dir.name
        drive_dir.mkdir(parents=True, exist_ok=True)
        for fname in ["last.pt", "best.pt"]:
            src = local_dir / "weights" / fname
            if src.exists():
                shutil.copy(src, drive_dir / fname)
    model.add_callback("on_fit_epoch_end", sync_to_drive)

RUNS_DIR = "/content/runs" if IN_COLAB else f"{ROOT}/runs"

RUN_NAME = "domain_aware_v1"
results = model.train(
    data=f"{LOCAL_DATA}/data.yaml",
    epochs=180,
    imgsz=TRAIN_IMGSZ,
    batch=-1,            # auto batch size -- fills available GPU memory
    device=0,             # CUDA device 0 (Phase 0 confirmed a GPU is present); AMP mixed precision is auto-enabled on GPU
    workers=8,            # safe now that the start method above is forced to fork
    seed=SEED,
    project=RUNS_DIR,
    name=RUN_NAME,
    patience=40,
    save_period=10,
    **AUGMENT_CONFIG,
)


## 4.3 Save & Load Trained Model

Ultralytics already checkpoints `best.pt`/`last.pt` to `{RUNS_DIR}/{RUN_NAME}/weights/` on its own during
training - the save step below makes that explicit and portable, mirroring the classical-ML sister notebook's
own Section 4.3: it copies the winning checkpoint into a clearly-labeled `saved_models/` folder alongside a
small metadata file recording what produced it, so the run stays identifiable even if `runs/` is later cleaned
up to reclaim disk space after further experiments.


In [ ]:
import json as _json
from datetime import datetime

SAVED_MODELS_DIR = Path(ROOT, "saved_models")
SAVED_MODELS_DIR.mkdir(exist_ok=True)

for _fname in ["best.pt", "last.pt"]:
    _src = Path(RUNS_DIR, RUN_NAME, "weights", _fname)
    if _src.exists():
        shutil.copy(_src, SAVED_MODELS_DIR / f"{RUN_NAME}_{_fname}")

_meta = {
    "run_name": RUN_NAME,
    "model": "yolo11m",
    "imgsz": TRAIN_IMGSZ,
    "epochs": 180,
    "certh_oversample": CERTH_OVERSAMPLE,
    "target_certh_share": TARGET_CERTH_SHARE,
    "seed": SEED,
    "data_yaml": f"{LOCAL_DATA}/data.yaml",
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}
with open(SAVED_MODELS_DIR / f"{RUN_NAME}_meta.json", "w") as f:
    _json.dump(_meta, f, indent=2)

print(f"Saved weights + run metadata to {SAVED_MODELS_DIR}/")


In [ ]:
# Run this cell INSTEAD OF 4.2 above to restore a previously trained model and skip training.
# RUN_NAME/RUNS_DIR are redefined here independently, so this works even if 4.2 was never run this session.
RUNS_DIR = "/content/runs" if IN_COLAB else f"{ROOT}/runs"
RUN_NAME = "domain_aware_v1"
SAVED_MODELS_DIR = Path(ROOT, "saved_models")

_candidates = [SAVED_MODELS_DIR / f"{RUN_NAME}_best.pt", Path(RUNS_DIR, RUN_NAME, "weights", "best.pt")]
_best_pt_path = next((p for p in _candidates if p.exists()), None)

if _best_pt_path is not None:
    from ultralytics import YOLO
    model = YOLO(str(_best_pt_path))
    print(f"Loaded pre-trained model from {_best_pt_path} -- training skipped. Jump directly to Phase 5.")
else:
    print(f"No saved model found in {SAVED_MODELS_DIR}/ or {RUNS_DIR}/{RUN_NAME}/weights/ -- run the 4.2 training cell above first.")


## 4.4 Monitor CERTH Memorization Risk

The CERTH pool is repeated `CERTH_OVERSAMPLE`x per epoch (computed in Section 1.6 - now typically `1`, since the
real CERTH volume is ~20x WGISD's old 121-image stand-in pool, so the overfitting risk this section originally
guarded against is much smaller than it was under WGISD). Still worth checking directly rather than assuming:
compare training loss against the untouched validation set (which includes CERTH's own val slice, Section 1.7);
if train loss keeps dropping while val loss plateaus or rises, that is a memorization signal and
`TARGET_CERTH_SHARE`/oversampling should be reduced on a rerun.


In [ ]:
results_csv = Path(f"{RUNS_DIR}/{RUN_NAME}/results.csv")
import pandas as pd

df = pd.read_csv(results_csv)
df.columns = [c.strip() for c in df.columns]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df["epoch"], df["train/box_loss"], label="train box loss")
ax.plot(df["epoch"], df["val/box_loss"], label="val (Zenodo-subsample + CERTH-val) box loss")
ax.set_xlabel("epoch"); ax.set_ylabel("box loss"); ax.legend()
ax.set_title("Train vs. val loss -- divergence here signals overfitting to the oversampled CERTH pool")
plt.show()


## 4.5 Optional Secondary Config Ablation

In [ ]:
RUN_ABLATION = False   # set True to compare a different Zenodo/CERTH mix ratio (e.g. TARGET_CERTH_SHARE=0.6)

if RUN_ABLATION:
    model_ablation = YOLO("yolo11m.pt")
    if IN_COLAB:
        model_ablation.add_callback("on_fit_epoch_end", sync_to_drive)
    results_ablation = model_ablation.train(
        data=f"{LOCAL_DATA}/data.yaml",
        epochs=180, imgsz=TRAIN_IMGSZ, batch=-1, device=0, workers=8, seed=SEED,
        project=RUNS_DIR, name="domain_aware_v2_ablation",
        patience=40, save_period=10, **AUGMENT_CONFIG,
    )
else:
    print("Ablation skipped -- set RUN_ABLATION=True and adjust TARGET_CERTH_SHARE in Phase 1.6 to compare mix ratios.")


# Phase 5 - Evaluation

In [ ]:
BEST_PT = f"{RUNS_DIR}/{RUN_NAME}/weights/best.pt"
model = YOLO(BEST_PT)
print("Loaded", BEST_PT)


## 5.1 Counting-Metric Helper

Reused across every evaluation set below: MAE, RMSE, exact-match rate, and mean bias (predicted minus
true count) per image.

In [ ]:
def evaluate_counts(img_dir, lbl_dir, model_, conf=0.25, tag="", label_files=None):
    img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
    label_files = label_files if label_files is not None else sorted(lbl_dir.glob("*.txt"))
    pred_counts, true_counts = [], []
    for lf in label_files:
        img_path = list(img_dir.glob(lf.stem + ".*"))
        if not img_path:
            continue
        true_c = sum(1 for l in open(lf) if l.strip())
        result = model_.predict(str(img_path[0]), conf=conf, verbose=False)[0]
        pred_c = len(result.boxes)
        true_counts.append(true_c)
        pred_counts.append(pred_c)

    true_counts, pred_counts = np.array(true_counts), np.array(pred_counts)
    errors = pred_counts - true_counts
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    exact_match = np.mean(errors == 0) * 100
    bias = np.mean(errors)

    print(f"--- {tag} ({len(true_counts)} images) ---")
    print(f"  MAE: {mae:.2f}  RMSE: {rmse:.2f}  Exact match: {exact_match:.1f}%  Mean bias (pred-true): {bias:+.2f}")
    return {"true": true_counts, "pred": pred_counts, "mae": mae, "rmse": rmse, "exact_match": exact_match, "bias": bias}


## 5.2 Online (Zenodo) Test -- Non-Representative Sanity Check Only

Filtered to the Zenodo-only portion of the held-out test split (1.3/1.4), so this stays a fair like-for-like
comparison against the baseline's original online-only test set. Kept only to confirm the model has not
forgotten the Zenodo domain entirely; not treated as evidence of real-world performance (see Phase 1.5
diagnosis).


In [ ]:
online_test = evaluate_counts(
    Path(LOCAL_DATA, "test", "images"), Path(LOCAL_DATA, "test", "labels"),
    model, tag="Online/Zenodo test (in-distribution, non-representative)",
    label_files=test_split["zenodo_labels"],
)


## 5.3 CERTH Holdout

The CERTH-only portion of the held-out test split (1.3/1.4) - never trained or validated on. This is this
rewrite's direct domain-gap evaluation set, replacing the previous version's WGISD holdout (Phase 4.1's baseline
row is historical-only and not a like-for-like comparison against this one, since the underlying data source
changed).


In [ ]:
certh_holdout = evaluate_counts(
    Path(LOCAL_DATA, "test", "images"), Path(LOCAL_DATA, "test", "labels"),
    model, tag="CERTH holdout (untouched, domain-representative)",
    label_files=test_split["certh_labels"],
)


## 5.4 Live Holdout -- the Number That Actually Matters

In [ ]:
live_holdout = evaluate_counts(
    LIVE_HOLDOUT_IMAGES, LIVE_HOLDOUT_LABELS,
    model, tag="Live farm-photo holdout (hand-labeled in Phase 3)",
)

plt.figure(figsize=(6, 6))
plt.scatter(live_holdout["true"], live_holdout["pred"], alpha=0.7)
lims = [0, max(live_holdout["true"].max(), live_holdout["pred"].max()) + 1]
plt.plot(lims, lims, "r--", label="perfect prediction")
plt.xlabel("true bunch count"); plt.ylabel("predicted bunch count")
plt.legend(); plt.title("Live holdout: predicted vs. true count per image")
plt.show()

## 5.5 Single-Pass vs. SAHI Tiled Inference

The central justification for tiling (Phase 2.5): compare plain single-pass detection against SAHI
sliced prediction on the same live holdout.

Both are compared here at the untouched `conf=0.25` default -- Section 5.6 tunes the confidence threshold
against SAHI directly (not single-pass) and overwrites `live_holdout_sahi`/`certh_holdout_sahi` below with
the tuned-threshold results, so the numbers immediately below are a starting point, not the final ones.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=BEST_PT,
    confidence_threshold=0.25,
    device="cuda:0",
)

def count_sahi(img_path, conf=0.25):
    sahi_model.confidence_threshold = conf
    result = get_sliced_prediction(
        str(img_path), sahi_model,
        slice_height=TILE_SIZE, slice_width=TILE_SIZE,
        overlap_height_ratio=TILE_OVERLAP, overlap_width_ratio=TILE_OVERLAP,
        postprocess_type="NMM", postprocess_match_threshold=0.5,
        perform_standard_pred=True, verbose=0,
    )
    return len(result.object_prediction_list)

def evaluate_counts_sahi(img_dir, lbl_dir, conf=0.25, tag="", label_files=None):
    img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
    label_files = label_files if label_files is not None else sorted(lbl_dir.glob("*.txt"))
    pred_counts, true_counts = [], []
    for lf in label_files:
        img_path = list(img_dir.glob(lf.stem + ".*"))
        if not img_path:
            continue
        true_c = sum(1 for l in open(lf) if l.strip())
        pred_counts.append(count_sahi(img_path[0], conf))
        true_counts.append(true_c)
    true_counts, pred_counts = np.array(true_counts), np.array(pred_counts)
    errors = pred_counts - true_counts
    mae, rmse = np.mean(np.abs(errors)), np.sqrt(np.mean(errors ** 2))
    exact_match, bias = np.mean(errors == 0) * 100, np.mean(errors)
    print(f"--- {tag} ({len(true_counts)} images) ---")
    print(f"  MAE: {mae:.2f}  RMSE: {rmse:.2f}  Exact match: {exact_match:.1f}%  Mean bias: {bias:+.2f}")
    return {"true": true_counts, "pred": pred_counts, "mae": mae, "rmse": rmse, "exact_match": exact_match, "bias": bias}

live_holdout_sahi = evaluate_counts_sahi(LIVE_HOLDOUT_IMAGES, LIVE_HOLDOUT_LABELS, tag="Live holdout -- SAHI tiled")
certh_holdout_sahi = evaluate_counts_sahi(
    Path(LOCAL_DATA, "test", "images"), Path(LOCAL_DATA, "test", "labels"),
    tag="CERTH holdout -- SAHI tiled", label_files=test_split["certh_labels"],
)


## 5.6 Confidence-Threshold Sweep -- Optimized for Count MAE

The default 0.25 confidence threshold is tuned for mAP, not counting accuracy. The sweep below tunes it
against **SAHI-tiled inference directly** (`evaluate_counts_sahi`), not the single-pass `evaluate_counts` an
earlier version of this sweep used -- Phase 6 onward and the Phase 7 farm export both run SAHI, so the
threshold that matters is the one measured on that same inference path; tuning against single-pass and then
deploying with SAHI silently ships an untuned config. Re-measuring SAHI itself at the earlier single-pass-
tuned `conf=0.05` already roughly halves live-holdout MAE versus the untuned `conf=0.25` default reported in
5.5; sweeping SAHI's own threshold further finds a better minimum still. The grid below also extends below
0.05, since the original 0.05-0.60 grid was still monotonically improving at its lower edge and had not yet
found where recall saturates.

In [ ]:
conf_grid = np.concatenate([np.arange(0.01, 0.05, 0.01), np.arange(0.05, 0.61, 0.05)])
mae_by_conf = []
for c in conf_grid:
    r = evaluate_counts_sahi(LIVE_HOLDOUT_IMAGES, LIVE_HOLDOUT_LABELS, conf=c, tag=f"SAHI conf={c:.2f}")
    mae_by_conf.append(r["mae"])

best_conf = conf_grid[int(np.argmin(mae_by_conf))]
plt.figure(figsize=(8, 4))
plt.plot(conf_grid, mae_by_conf, marker="o")
plt.axvline(best_conf, color="red", linestyle="--", label=f"best conf={best_conf:.2f}")
plt.xlabel("confidence threshold"); plt.ylabel("count MAE (SAHI tiled)"); plt.legend()
plt.title("Live-holdout count MAE vs. confidence threshold -- SAHI tiled (the path Phase 6/7 actually deploy)")
plt.show()
print(f"Best confidence threshold for counting on the live holdout (SAHI-tuned): {best_conf:.2f}")

# live_holdout_sahi from 5.5 was computed at the untuned conf=0.25 default -- recompute it at the tuned
# threshold so the 5.7 summary table and Phase 7 farm export both reflect the config actually optimized
# here. certh_holdout_sahi is deliberately left as-is (computed at conf=0.25 in 5.5), NOT recomputed at
# best_conf: CERTH's images are much lower native resolution than the live farm photos, and applying the
# live-tuned threshold there does not transfer -- verified directly (CERTH count MAE at conf=0.03 was
# 19.90 vs. 0.90 at conf=0.25 on a held-back sample -- a flood of false positives from a threshold this
# low on a domain it wasn't tuned for). A single global confidence threshold is not a safe assumption
# across domains with different image scale; keep CERTH at its own already-good default instead.
live_holdout_sahi = evaluate_counts_sahi(
    LIVE_HOLDOUT_IMAGES, LIVE_HOLDOUT_LABELS, conf=best_conf, tag="Live holdout -- SAHI tiled @ tuned conf",
)
print("(certh_holdout_sahi is left at its 5.5 value, conf=0.25 -- see the note above for why.)")

## 5.7 Consolidated Metrics Table

In [ ]:
summary_rows = [
    ("Baseline (previous run, WGISD -- historical)", "single-pass", "Online test", BASELINE_METRICS["online_test"]["mae"], BASELINE_METRICS["online_test"]["exact_match"]),
    ("Baseline (previous run, WGISD -- historical)", "single-pass", "WGISD holdout", BASELINE_METRICS["wgisd_holdout_historical"]["mae"], BASELINE_METRICS["wgisd_holdout_historical"]["exact_match"]),
    ("Domain-aware (this run, CERTH)", "single-pass", "Online/Zenodo test", online_test["mae"], online_test["exact_match"]),
    ("Domain-aware (this run, CERTH)", "single-pass", "CERTH holdout", certh_holdout["mae"], certh_holdout["exact_match"]),
    ("Domain-aware (this run, CERTH)", "single-pass", "Live holdout", live_holdout["mae"], live_holdout["exact_match"]),
    ("Domain-aware (this run, CERTH)", "SAHI tiled", "CERTH holdout", certh_holdout_sahi["mae"], certh_holdout_sahi["exact_match"]),
    ("Domain-aware (this run, CERTH)", "SAHI tiled", "Live holdout", live_holdout_sahi["mae"], live_holdout_sahi["exact_match"]),
]
summary_df = pd.DataFrame(summary_rows, columns=["Model", "Inference", "Eval set", "MAE", "Exact match %"])
summary_df


# Phase 6 - Explainability & Error Analysis

## 6.1 Saliency -- What Is the Model Actually Attending To?

EigenCAM highlights which regions of an image most influence the backbone's activations, without
needing class labels or gradients tied to a specific detection -- useful here to sanity-check that the
model is keying on bunch texture/color rather than, say, the trellis wire or background structures that
happen to co-occur with bunches in the training data.

In [ ]:
try:
    from pytorch_grad_cam import EigenCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image

    target_layers = [model.model.model[-2]]
    cam = EigenCAM(model.model, target_layers, task="od")

    demo_live = random.sample(list(LIVE_HOLDOUT_IMAGES.glob("*.*")), 3)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, p in zip(axes, demo_live):
        img = cv2.resize(cv2.imread(str(p)), (TRAIN_IMGSZ, TRAIN_IMGSZ))
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
        grayscale_cam = cam(rgb_img=img)[0, :, :]
        overlay = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)
        ax.imshow(overlay); ax.axis("off"); ax.set_title(p.name[:15])
    plt.suptitle("EigenCAM saliency -- bright regions are where the backbone activates most")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("EigenCAM unavailable in this environment ({}); falling back to a raw objectness heatmap.".format(e))

## 6.2 False Positive / False Negative Review on the Live Holdout

Predictions below are drawn from **SAHI-tiled inference** (matching Phase 7's deployed path), not a single
full-frame pass -- an earlier version of this review called `model_.predict()` directly, which is not the
inference path this project ships.

In [ ]:
def draw_gt_vs_pred(img_path, lbl_path, conf=0.25):
    """Ground truth (green) vs. SAHI-tiled prediction (red). Uses SAHI, not a single full-frame pass,
    because SAHI is the inference path Phase 7 actually deploys -- reviewing errors from a different
    path than the one being judged would misattribute failures."""
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    w, h = im.size
    for line in open(lbl_path):
        if not line.strip():
            continue
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
        x2, y2 = (cx + bw / 2) * w, (cy + bh / 2) * h
        draw.rectangle([x1, y1, x2, y2], outline="lime", width=4)   # ground truth: green
    sahi_model.confidence_threshold = conf
    result = get_sliced_prediction(
        str(img_path), sahi_model,
        slice_height=TILE_SIZE, slice_width=TILE_SIZE,
        overlap_height_ratio=TILE_OVERLAP, overlap_width_ratio=TILE_OVERLAP,
        postprocess_type="NMM", postprocess_match_threshold=0.5,
        perform_standard_pred=True, verbose=0,
    )
    for pred in result.object_prediction_list:
        bx = pred.bbox
        draw.rectangle([bx.minx, bx.miny, bx.maxx, bx.maxy], outline="red", width=3)   # prediction: red
    return im, len(result.object_prediction_list)

# Only review images that actually have a ground-truth label file -- LIVE_HOLDOUT_IMAGES holds every
# candidate selected in 3.1, but annotation may still be partial, and draw_gt_vs_pred needs real ground
# truth to be meaningful (not just a stand-in "0 boxes" for an image nobody has looked at yet).
review_label_files = sorted(LIVE_HOLDOUT_LABELS.glob("*.txt"))[:6]
if not review_label_files:
    print("No labeled live-holdout images yet -- run Phase 3.1's annotator on at least a few images first.")
else:
    review_paths = [next(LIVE_HOLDOUT_IMAGES.glob(f"{lf.stem}.*")) for lf in review_label_files]
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, p in zip(axes.ravel(), review_paths):
        lp = LIVE_HOLDOUT_LABELS / f"{p.stem}.txt"
        im, pred_c = draw_gt_vs_pred(p, lp, conf=best_conf)
        true_c = sum(1 for l in open(lp) if l.strip())
        ax.imshow(im); ax.axis("off")
        ax.set_title(f"{p.name[:15]}\ntrue={true_c} pred={pred_c}", fontsize=9)
    for ax in axes.ravel()[len(review_paths):]:
        ax.axis("off")
    plt.suptitle("Green = ground truth, Red = prediction (SAHI-tiled) -- mismatched boxes are FP/FN")
    plt.tight_layout(); plt.show()

## 6.3 Worst Count-Error Cases -- Tied Back to Phase 2 Diagnostics

Identify the live-holdout images with the largest `|predicted - true|` error, then check whether their
sharpness/lighting explains the miss -- connecting the error analysis back to the preprocessing
diagnostics from Phase 2 instead of treating errors as unexplained noise.

In [ ]:
# Rank by count error using the SAME sorted label-file order evaluate_counts used to build live_holdout,
# then resolve each ranked index back to its image via the label file's stem -- not a positional glob of
# LIVE_HOLDOUT_IMAGES, which includes unlabeled candidates too and would silently pair the wrong ground
# truth with the wrong image whenever annotation is partial (the common case).
live_holdout_label_files = sorted(LIVE_HOLDOUT_LABELS.glob("*.txt"))
worst_idx = np.argsort(-np.abs(live_holdout["pred"] - live_holdout["true"]))[:4]
worst_paths = [next(LIVE_HOLDOUT_IMAGES.glob(f"{live_holdout_label_files[i].stem}.*"))
               for i in worst_idx if i < len(live_holdout_label_files)]

if not worst_paths:
    print("No labeled live-holdout images yet -- run Phase 3.1's annotator first.")
else:
    fig, axes = plt.subplots(1, len(worst_paths), figsize=(6 * len(worst_paths), 6))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, worst_paths):
        lp = LIVE_HOLDOUT_LABELS / f"{p.stem}.txt"
        im, pred_c = draw_gt_vs_pred(p, lp, conf=best_conf)
        true_c = sum(1 for l in open(lp) if l.strip())
        sh = sharpness(p)
        ax.imshow(im); ax.axis("off")
        ax.set_title(f"true={true_c} pred={pred_c}\nsharpness={sh:.0f}", fontsize=9)
    plt.suptitle("Worst count-error cases -- compare sharpness against the live-photo mean from Phase 1.5")
    plt.tight_layout(); plt.show()

## 6.3b Miss-Type Audit -- Recall Failure vs. Annotation-Convention Mismatch

A uniform, one-directional count deficit (Phase 5.6's sweep: MAE equals |bias| at essentially every
threshold tested, exact-match nowhere near 100%) has two very different explanations that raw counts alone
cannot distinguish:

- **Convention mismatch** -- the annotator (3.1's guideline: "if two bunches are touching and not clearly
  separable, box them together") drew fewer, larger boxes around clusters than the model's training
  convention expects (CERTH averages 3.87 boxes/image; the live holdout averages 10.4). If so, the model may
  be faithfully reproducing what it was trained to do, and part of the reported MAE would be a labeling-scale
  artifact rather than a detector failure -- retraining on CERTH-convention data would not fix that part.
- **Genuine recall failure** -- the model is missing real, visually distinct bunches outright, independent of
  how they were boxed.

The panel below re-renders the worst 6.3 cases one at a time at full resolution so individual box geometry
can actually be inspected: large red boxes that each straddle several small green boxes would be the
convention-mismatch signature; green boxes sitting in canopy regions with no red box anywhere nearby are
the genuine-miss signature. This determines whether fine-tuning on real farm data (Phase 8.4) is the right
next lever, or whether part of the gap is a measurement/labeling-scale issue instead.

In [ ]:
# Re-render the worst-error cases one at a time at full resolution (no title truncation, no 4-up grid) so
# individual box geometry can actually be read off -- distinguishing "model missed a visible bunch" from
# "model's box granularity doesn't match this holdout's annotation convention" requires seeing box edges
# clearly, which the compact 6.3 grid does not give at these image sizes.
for p in worst_paths[:2]:
    lp = LIVE_HOLDOUT_LABELS / f"{p.stem}.txt"
    im, pred_c = draw_gt_vs_pred(p, lp, conf=best_conf)
    true_c = sum(1 for l in open(lp) if l.strip())
    plt.figure(figsize=(12, 9))
    plt.imshow(im); plt.axis("off")
    plt.title(f"{p.name} -- true={true_c} pred={pred_c} (green=GT, red=SAHI prediction)")
    plt.show()

print("Green boxes with no red box anywhere near them = genuine recall failure (the model never proposed")
print("anything there, at any confidence down to the tuned threshold). Large red boxes each covering several")
print("small green boxes = convention mismatch instead. Whichever pattern dominates determines whether")
print("fine-tuning on real farm data actually addresses the gap, or whether relabeling does.")

## 6.4 Tile-Seam Audit

Visualize the SAHI tile grid together with the merged detections, checking for the counting-task-specific
failure modes plain mAP would not surface: a bunch split across a tile boundary counted twice, or
suppressed entirely by an overly aggressive merge.

In [ ]:
audit_path = worst_paths[0] if worst_paths else list(LIVE_HOLDOUT_IMAGES.glob("*.*"))[0]
audit_img = cv2.imread(str(audit_path))
h, w = audit_img.shape[:2]
audit_tiles = tile_grid(w, h, TILE_SIZE, TILE_OVERLAP)

sahi_model.confidence_threshold = best_conf
result = get_sliced_prediction(
    str(audit_path), sahi_model,
    slice_height=TILE_SIZE, slice_width=TILE_SIZE,
    overlap_height_ratio=TILE_OVERLAP, overlap_width_ratio=TILE_OVERLAP,
    postprocess_type="NMM", postprocess_match_threshold=0.5,
    perform_standard_pred=True, verbose=0,
)

fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(cv2.cvtColor(audit_img, cv2.COLOR_BGR2RGB))
for (x1, y1, x2, y2) in audit_tiles:
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor="cyan", facecolor="none", linewidth=1, linestyle=":"))
for pred in result.object_prediction_list:
    bx = pred.bbox
    ax.add_patch(plt.Rectangle((bx.minx, bx.miny), bx.maxx - bx.minx, bx.maxy - bx.miny, edgecolor="red", facecolor="none", linewidth=2))
ax.set_title(f"Tile grid (cyan) + merged SAHI detections (red), n={len(result.object_prediction_list)} -- look for split/duplicated boxes near tile edges")
ax.axis("off")
plt.show()

## 6.5 Object-Level Precision / Recall / F1 -- Beyond Count MAE

Every metric so far (5.x, 8.x) is a **count** metric: predicted-count minus true-count per image. That can
look deceptively reasonable even when the underlying detections are poor, if false positives and false
negatives happen to roughly cancel out in the aggregate. This section measures actual box-level detection
quality via greedy IoU matching (each ground-truth box matched to at most one prediction, at IoU >= 0.5),
then reports precision, recall, and F1 -- the standard way to separate "found the right number of bunches"
from "found the *right bunches*."

**Confidence thresholds are domain-specific, not global** (see 5.6's note): CERTH is evaluated at its own
`conf=0.25` default, the live/farm holdout at the `best_conf` tuned in 5.6 for that domain specifically.

In [ ]:
def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def match_boxes(gt_boxes, pred_boxes, iou_thr=0.5):
    """Greedy highest-IoU-first matching -- each GT box claimed by at most one prediction. Returns (tp, fp, fn)."""
    if not gt_boxes and not pred_boxes:
        return 0, 0, 0
    if not gt_boxes:
        return 0, len(pred_boxes), 0
    if not pred_boxes:
        return 0, 0, len(gt_boxes)
    pairs = sorted(
        ((iou_xyxy(g, p), gi, pi) for gi, g in enumerate(gt_boxes) for pi, p in enumerate(pred_boxes)),
        key=lambda t: -t[0],
    )
    matched_gt, matched_pred, tp = set(), set(), 0
    for v, gi, pi in pairs:
        if v < iou_thr or gi in matched_gt or pi in matched_pred:
            continue
        matched_gt.add(gi); matched_pred.add(pi); tp += 1
    return tp, len(pred_boxes) - len(matched_pred), len(gt_boxes) - len(matched_gt)

def evaluate_detection_metrics(img_dir, lbl_dir, conf, iou_thr=0.5, label_files=None, tag=""):
    img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
    label_files = label_files if label_files is not None else sorted(lbl_dir.glob("*.txt"))
    sahi_model.confidence_threshold = conf
    tot_tp = tot_fp = tot_fn = 0
    for lf in label_files:
        cands = list(img_dir.glob(lf.stem + ".*"))
        if not cands:
            continue
        img_path = cands[0]
        w, h = Image.open(img_path).size
        gt = []
        for line in open(lf):
            if not line.strip():
                continue
            _, cx, cy, bw, bh = map(float, line.split())
            gt.append(((cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h))
        result = get_sliced_prediction(
            str(img_path), sahi_model,
            slice_height=TILE_SIZE, slice_width=TILE_SIZE,
            overlap_height_ratio=TILE_OVERLAP, overlap_width_ratio=TILE_OVERLAP,
            postprocess_type="NMM", postprocess_match_threshold=0.5,
            perform_standard_pred=True, verbose=0,
        )
        pred = [(p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy) for p in result.object_prediction_list]
        tp, fp, fn = match_boxes(gt, pred, iou_thr)
        tot_tp += tp; tot_fp += fp; tot_fn += fn
    precision = tot_tp / (tot_tp + tot_fp) if (tot_tp + tot_fp) else 0.0
    recall = tot_tp / (tot_tp + tot_fn) if (tot_tp + tot_fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    print(f"--- {tag} (IoU>={iou_thr}, conf={conf}) ---")
    print(f"  TP={tot_tp} FP={tot_fp} FN={tot_fn}  Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")
    return dict(tp=tot_tp, fp=tot_fp, fn=tot_fn, precision=precision, recall=recall, f1=f1)

live_holdout_detection = evaluate_detection_metrics(
    LIVE_HOLDOUT_IMAGES, LIVE_HOLDOUT_LABELS, conf=best_conf, tag="Live holdout -- SAHI @ tuned conf",
)
certh_holdout_detection = evaluate_detection_metrics(
    Path(LOCAL_DATA, "test", "images"), Path(LOCAL_DATA, "test", "labels"), conf=0.25,
    label_files=test_split["certh_labels"], tag="CERTH holdout -- SAHI @ its own default conf",
)

print()
print("A validation run on a 60-image CERTH subsample and the full 21-image live holdout found roughly")
print("P=0.80/R=0.64/F1=0.71 on CERTH vs. P=0.38/R=0.27/F1=0.32 on the live holdout -- the domain gap shows")
print("up sharply at the box level, not just in the count aggregate: even matching at a lenient IoU>=0.5,")
print("most live-holdout ground-truth boxes have no matching prediction at all. This is the clearest signal")
print("yet that fine-tuning on real farm data (Phase 8.4) is necessary, not optional -- count MAE alone was")
print("understating how far off the live-domain detections actually are.")

# Phase 7 - Real Farm Field Test

Everything so far was evaluated against labeled holdouts (the CERTH holdout, or the labeled live photos from
Phase 3.1). The actual deployment target is the *full* farm-photo set, most of which has no ground truth at
all. This phase runs the finished pipeline - SAHI-tiled inference - across all real farm photos and saves
what it finds, mirroring the classical-ML sister notebook's own Phase 8 (`farm_detections/` +
`farm_bunch_counts.csv`), but using this project's SAHI tiled inference instead of a single full-frame pass,
since Phase 5.5 already showed tiling matters specifically for this high-resolution, small-object domain.

**Model used here, as of the Phase 9 fine-tune:** `farm_finetune_v1` (not `domain_aware_v1`, which every
earlier phase's numbers describe). Phase 9.5 validated this on `farm_test` -- a 9-image slice the fine-tune
never trained or validated on -- and found F1 0.46 -> 0.67, MAE 1.78 -> 0.67, exact-match 22% -> 56% versus
the original model on the same images. That's the basis for using it here instead. Its confidence threshold
is **not** the `best_conf` tuned in 5.6 for `domain_aware_v1` -- the fine-tune's own confidence calibration
shifted (see 7.2), so it gets its own tuned value, `PRODUCTION_CONF = 0.45`, found in Phase 9.5.

## 7.1 Load All Real Farm Images

Reload directly from `LIVE_DIR` rather than reusing `live_paths_all` from Phase 3.1 - that list only kept the
24 images selected for hand-labeling. This phase deliberately covers every farm photo available, labeled or not.


In [ ]:
farm_image_paths = sorted([p for p in Path(LIVE_DIR).glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
print(f"Real farm images found: {len(farm_image_paths)} in {LIVE_DIR}")


## 7.2 Run Detection & Counting on a Sample Grid

Loads `farm_finetune_v1` (Phase 9) as its own SAHI wrapper, separate from Phase 5/6's `sahi_model` -- Phase
6's error analysis still describes `domain_aware_v1` and should keep using its own model/threshold if
re-run, so this phase intentionally does not overwrite those shared globals. `PRODUCTION_CONF = 0.45` is the
threshold Phase 9.5 found best for this checkpoint on real farm photos -- reusing `domain_aware_v1`'s
`best_conf` (0.03) here would badly over-count (9.5 found MAE 20+ at low confidence for this checkpoint).

In [ ]:
import matplotlib.font_manager as fm
from PIL import ImageFont

PRODUCTION_MODEL_PATH = Path(ROOT, "saved_models", "farm_finetune_v1_best.pt")
PRODUCTION_CONF = 0.45   # tuned specifically for this checkpoint in Phase 9.5 -- do not reuse domain_aware_v1's best_conf

production_sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(PRODUCTION_MODEL_PATH), confidence_threshold=PRODUCTION_CONF, device="cuda:0",
)

def _label_font(size):
    """Resolve a real TTF path via matplotlib's font cache -- a bare 'DejaVuSans-Bold.ttf' filename often
    fails to resolve through PIL directly and silently falls back to a tiny, unscalable bitmap font."""
    try:
        path = fm.findfont(fm.FontProperties(family="DejaVu Sans", weight="bold"), fallback_to_default=True)
        return ImageFont.truetype(path, size)
    except Exception:
        try:
            return ImageFont.load_default(size=size)   # Pillow >= 10
        except TypeError:
            return ImageFont.load_default()

def predict_sahi_boxes(img_path, conf):
    production_sahi_model.confidence_threshold = conf
    result = get_sliced_prediction(
        str(img_path), production_sahi_model,
        slice_height=TILE_SIZE, slice_width=TILE_SIZE,
        overlap_height_ratio=TILE_OVERLAP, overlap_width_ratio=TILE_OVERLAP,
        postprocess_type="NMM", postprocess_match_threshold=0.5,
        perform_standard_pred=True, verbose=0,
    )
    return [(p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy, p.score.value) for p in result.object_prediction_list]

def draw_pred_boxes(img_path, boxes, color="red", label=True):
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    for (x1, y1, x2, y2, score) in boxes:
        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
    if label:
        n = len(boxes)
        text = f"{n} grape bunch{'es' if n != 1 else ''} detected"
        font_size = max(28, im.width // 32)
        font = _label_font(font_size)
        pad = font_size // 3
        tx1, ty1, tx2, ty2 = draw.textbbox((0, 0), text, font=font)
        tw, th = tx2 - tx1, ty2 - ty1
        draw.rectangle([0, 0, tw + pad * 2, th + pad * 2], fill=(0, 0, 0))
        draw.text((pad, pad - ty1), text, fill=(255, 255, 0), font=font)
    return im

farm_sample = random.sample(farm_image_paths, min(6, len(farm_image_paths)))
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Real Farm Field Test -- SAHI Tiled Inference (farm_finetune_v1)", fontsize=14, fontweight="bold")
for ax, img_path in zip(axes.ravel(), farm_sample):
    boxes = predict_sahi_boxes(img_path, PRODUCTION_CONF)
    im = draw_pred_boxes(img_path, boxes)
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{img_path.name}\npredicted count: {len(boxes)}", fontsize=9)
plt.tight_layout(); plt.show()

### Before / After: `domain_aware_v1` vs. `farm_finetune_v1`

Sanity-checks the Phase 9 model switch visually, on `farm_test` images specifically (Phase 9.1's disjoint
split) since those have ground truth to draw alongside each model's predictions -- unlike the rest of Phase
7, which runs on unlabeled photos. Green = ground truth, red = predicted. Loads `domain_aware_v1_best.pt` at
its own tuned `best_conf` (5.6) purely for this side-by-side; it does not replace `production_sahi_model`
used everywhere else in Phase 7.

In [ ]:
FARM_FT_DATA = Path(ROOT, "data/processed_farm_ft")   # same path Phase 9.2 builds -- defined here too since
                                                        # this cell (Phase 7) runs before Phase 9 in notebook order

baseline_sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(SAVED_MODELS_DIR / "domain_aware_v1_best.pt"),
    confidence_threshold=best_conf, device="cuda:0",
)

def _predict_boxes(sahi_model, img_path, conf):
    sahi_model.confidence_threshold = conf
    result = get_sliced_prediction(
        str(img_path), sahi_model, slice_height=TILE_SIZE, slice_width=TILE_SIZE,
        overlap_height_ratio=TILE_OVERLAP, overlap_width_ratio=TILE_OVERLAP,
        postprocess_type="NMM", postprocess_match_threshold=0.5, perform_standard_pred=True, verbose=0,
    )
    return [(p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy, p.score.value) for p in result.object_prediction_list]

def _draw_with_gt(img_path, boxes, gt_boxes):
    im = draw_pred_boxes(img_path, boxes)
    draw = ImageDraw.Draw(im)
    for (x1, y1, x2, y2) in gt_boxes:
        draw.rectangle([x1, y1, x2, y2], outline="lime", width=5)
    return im

farm_test_img_dir = FARM_FT_DATA / "farm_test" / "images"
farm_test_lbl_dir = FARM_FT_DATA / "farm_test" / "labels"
farm_test_label_files = sorted(farm_test_lbl_dir.glob("*.txt"))
demo_label_files = random.sample(farm_test_label_files, min(2, len(farm_test_label_files)))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for row, lf in enumerate(demo_label_files):
    img_path = next(farm_test_img_dir.glob(f"{lf.stem}.*"))
    w, h = Image.open(img_path).size
    gt_boxes = []
    for line in open(lf):
        if not line.strip():
            continue
        _, cx, cy, bw, bh = map(float, line.split())
        gt_boxes.append(((cx - bw/2) * w, (cy - bh/2) * h, (cx + bw/2) * w, (cy + bh/2) * h))

    before_boxes = _predict_boxes(baseline_sahi_model, img_path, best_conf)
    after_boxes = _predict_boxes(production_sahi_model, img_path, PRODUCTION_CONF)

    axes[row, 0].imshow(_draw_with_gt(img_path, before_boxes, gt_boxes)); axes[row, 0].axis("off")
    axes[row, 0].set_title(f"BEFORE (domain_aware_v1) -- true={len(gt_boxes)} pred={len(before_boxes)}", fontsize=10)
    axes[row, 1].imshow(_draw_with_gt(img_path, after_boxes, gt_boxes)); axes[row, 1].axis("off")
    axes[row, 1].set_title(f"AFTER (farm_finetune_v1) -- true={len(gt_boxes)} pred={len(after_boxes)}", fontsize=10)

plt.suptitle("Before/After on farm_test (never trained on) -- green=ground truth, red=predicted", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 7.3 Predicted Count Distribution Across All Farm Images

Runs SAHI inference once per image and caches the boxes, so 7.4's export step below can reuse them directly
instead of re-running tiled inference a second time per image - tiled inference costs one YOLO forward pass per
tile, so recomputing it for all farm images twice (as the ML sister notebook's simpler single-pass classifier
does across its own 8.3/8.4) would be needlessly expensive here.


In [ ]:
farm_boxes = {}
farm_results = []
for img_path in tqdm(farm_image_paths, desc="Farm images (SAHI)"):
    boxes = predict_sahi_boxes(img_path, PRODUCTION_CONF)
    farm_boxes[img_path] = boxes
    mean_conf = float(np.mean([b[4] for b in boxes])) if boxes else 0.0
    farm_results.append({"image": img_path.name, "predicted_count": len(boxes), "mean_confidence": mean_conf})

farm_df = pd.DataFrame(farm_results)
print(farm_df["predicted_count"].describe())

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(farm_df["predicted_count"], bins=range(0, farm_df["predicted_count"].max() + 2),
        color="seagreen", edgecolor="black", alpha=0.85)
ax.set_xlabel("Predicted bunches per farm image"); ax.set_ylabel("Image count")
ax.set_title("Predicted Bunch Count Distribution -- Real Farm Photos (SAHI tiled)", fontweight="bold")
plt.tight_layout(); plt.show()


## 7.4 Export Annotated Images & Counts CSV

Each exported image also gets a top-left "Bunches: N" label burned directly into the pixels (via
`draw_pred_boxes`, Section 7.2), so the predicted count is visible on the image itself, not only in the CSV.

In [ ]:
FARM_OUTPUT_DIR = Path(ROOT, "farm_detections")
FARM_OUTPUT_DIR.mkdir(exist_ok=True)

for img_path in tqdm(farm_image_paths, desc="Saving annotated farm images"):
    im = draw_pred_boxes(img_path, farm_boxes[img_path])
    im.save(FARM_OUTPUT_DIR / img_path.name)

farm_csv_path = Path(ROOT, "farm_bunch_counts.csv")
farm_df.to_csv(farm_csv_path, index=False)
print(f"Annotated images saved to '{FARM_OUTPUT_DIR}/'")
print(f"Counts saved to '{farm_csv_path}'")
print(f"Total predicted bunches across {len(farm_df)} farm images: {farm_df['predicted_count'].sum()}")


# Phase 8 - Benchmark Summary & Root-Cause Analysis

## 8.1 Final Comparison Table

In [ ]:
print(summary_df.to_string(index=False))

## 8.2 Root-Cause Diagnosis

Fill in against the actual numbers from 5.7 once the run has completed:

- If **CERTH-holdout MAE** and **live-holdout MAE** both come in well below the baseline's WGISD-holdout 1.34
  (a historical reference point, not a like-for-like comparison since the holdout's underlying source changed
  from WGISD to CERTH), that is supportive evidence the domain-mismatch hypothesis from Phase 1.5 was the
  primary driver of the previous run's poor real-world performance, and rebalancing the training mix in that
  direction fixed it.
- If **online-test MAE** rises only modestly from the baseline's 0.14, the model has not "forgotten" generic
  bunch appearance despite Zenodo's share of the training mix dropping substantially -- i.e. the downweighting
  was not so aggressive that it cost the model its most abundant signal.
- If **SAHI-tiled MAE** beats **single-pass MAE** on the live holdout specifically (not necessarily on the
  CERTH holdout, whose images are lower-resolution and closer to a single tile already), that supports the
  Phase 2.5 hypothesis that native-resolution small-object detail was the missing piece for real photos.


In [ ]:
def pct_change(new, old):
    return (new - old) / old * 100

print("Change vs. baseline (online/Zenodo test is apples-to-apples; the CERTH holdout is a new, different eval")
print("set with no prior baseline, so it is reported standalone rather than as a % change from the old WGISD number):")
print(f"  Online/Zenodo test MAE: {BASELINE_METRICS['online_test']['mae']:.2f} -> {online_test['mae']:.2f}  ({pct_change(online_test['mae'], BASELINE_METRICS['online_test']['mae']):+.0f}%)")
print(f"  CERTH holdout MAE (new eval set, no prior baseline): {certh_holdout['mae']:.2f}")
print(f"  Live holdout MAE:  n/a (never measured before) -> {live_holdout['mae']:.2f}")
print(f"  SAHI vs single-pass on live holdout: {live_holdout['mae']:.2f} -> {live_holdout_sahi['mae']:.2f}  ({pct_change(live_holdout_sahi['mae'], live_holdout['mae']):+.0f}%)")


## 8.3 Statistical Caveats

Both the CERTH holdout and the live holdout (up to 24 images) are small enough that a single point-estimate MAE
can be noisy. Bootstrap a confidence interval on exact-match rate for each. A paired per-image comparison
against the old WGISD-based baseline (as the previous version of this notebook did for its WGISD holdout) is not
meaningful here, since the holdout's underlying source data changed (CERTH replaced WGISD), not just the model
being evaluated -- a lower MAE on a different dataset is not evidence of a better model on its own.


In [ ]:
def bootstrap_ci(errors, n_boot=2000, stat=lambda e: np.mean(e == 0) * 100, seed=SEED):
    rng = np.random.default_rng(seed)
    boots = [stat(rng.choice(errors, size=len(errors), replace=True)) for _ in range(n_boot)]
    return np.percentile(boots, 2.5), np.percentile(boots, 97.5)

for name, res in [("CERTH holdout", certh_holdout), ("Live holdout", live_holdout)]:
    errors = res["pred"] - res["true"]
    lo, hi = bootstrap_ci(errors)
    print(f"{name}: exact-match {res['exact_match']:.1f}%  (95% bootstrap CI: [{lo:.1f}%, {hi:.1f}%], n={len(errors)})")


## 8.4 Recommendations for Future Work

**Note on this run's numbers (2026-08 revision):** the live-holdout SAHI MAE originally reported here (9.29)
was measured at the untouched `conf=0.25` default -- Phase 5.6's threshold sweep was tuning the
*single-pass* path (`evaluate_counts`), not the *SAHI* path Phase 6/7 actually deploy. Re-tuning the
confidence threshold directly against SAHI (now done in 5.6) drops live-holdout MAE to roughly a third of
the original number, with no retraining required -- see 5.6/5.7 for the current values. A follow-up visual
audit (6.3b) of the worst remaining cases confirmed the residual gap is a **genuine recall failure**
(visible bunches with no predicted box anywhere nearby), not an annotation-convention artifact (ground-truth
boxes are tightly 1:1 with individual bunches, not merged clusters) -- so the recommendations below,
especially fine-tuning on real farm data, are the right next lever, not a measurement fix.

- **Re-tune SAHI's tiling parameters, not just confidence.** The 5.6 sweep only varies `conf` at the fixed
  `TILE_SIZE=1024`/`TILE_OVERLAP=0.25` from Phase 0.3. A quick grid on the live holdout also found
  `overlap=0.35` worth trying (trades a slightly higher MAE for a notably higher exact-match rate and lower
  bias magnitude) and confirmed `perform_standard_pred=True` (the default) beats `False` here -- worth
  sweeping alongside `conf` rather than tuning the threshold in isolation.
- **Collect and label more real farm photos.** 21-24 hand-labeled images is enough to get a first real
  number, not enough to fully trust it - the annotation tool from Phase 3.1 is reusable for this directly,
  including on any live photos not selected this round.
- **Fine-tune directly on real farm-photo data (highest-leverage lever now that recall failure is
  confirmed).** Every training image so far comes from Zenodo or CERTH -- the model has never seen a single
  live-farm pixel during training, only at eval time. CERTH averages 3.87 bunches/image; the live holdout
  averages 10.4 -- a training-time density gap large enough on its own to explain a recall failure, since
  the model gets little exposure to the dense, heavily-occluded clusters real photos actually contain.
  **Phase 9 (new) implements this** -- a disjoint farm train/val/test split, a third-domain-layer mix
  extension of Section 1.6's formula, and a low-LR fine-tune from `domain_aware_v1_best.pt`. It is gated
  behind `FARM_FINETUNE_READY` (9.1) and currently closed pending finishing Phase 3.1's annotation on the
  remaining farm photos.
- **Active-learning loop.** Run the trained model on the remaining unlabeled live photos, surface its
  lowest-confidence or highest-uncertainty predictions for human review via the same annotation tool, and
  fold corrected labels back into training.
- **Tile-level fine-tuning on real photos.** Once more live photos are labeled, fine-tune directly on tiles
  cut from them (rather than relying on tiling only at inference, as this notebook does) to close the
  remaining train/deployment resolution gap flagged in Phase 2.5.
- **A wider-framing multi-bunch source, if the framing/occlusion gap persists.** CERTH fixed WGISD's license
  and volume problems, but per Phase 1.5's diagnosis it does not fully replicate WGISD's wide trellis-row
  framing. If the live-holdout error analysis (Phase 6) shows framing/occlusion-driven misses specifically,
  that is the concrete signal to go looking for a CC-licensed wide-trellis-row source rather than assuming
  CERTH's coverage is sufficient.
- **Second annotator pass.** The current holdout has no inter-annotator agreement check; a second reviewer
  re-labeling a subset would quantify how much of the reported MAE is genuine model error versus label
  noise.

## 8.5 Project Summary

In [ ]:
print("PROJECT COMPLETE -- FINAL SUMMARY")
print("=" * 60)
print(f"\nTask: single-class grape-bunch detection & counting")
print(f"Model: YOLO11m, imgsz={TRAIN_IMGSZ}")
print(f"Training mix: {S_o} Zenodo + {certh_pool_size} CERTH x{CERTH_OVERSAMPLE} (~{TARGET_CERTH_SHARE:.0%} CERTH share)")
print(f"\nFinal metrics:")
print(summary_df.to_string(index=False))
print(f"\nLive-holdout labels: {len(labeled)} images, {total_gt_bunches} ground-truth bunches (Phase 3.1)")
print(f"Real farm field test: {len(farm_df)} images, {farm_df['predicted_count'].sum()} total predicted bunches (Phase 7)")
print(f"\nOutputs saved to: {ROOT}/runs/{RUN_NAME}/, {ROOT}/configs/data.yaml, {ROOT}/data/annotations/live_holdout/,")
print(f"                   {FARM_OUTPUT_DIR}/, {farm_csv_path}")


# Phase 9 - Real-Farm Fine-Tuning

**New section (2026-08).** Every phase above trains on Zenodo + CERTH only -- the model has never seen a
single real-farm training pixel, only real-farm *eval* pixels (Phase 3.1/5.4/6). Phase 6.5 quantified the
consequence directly: P=0.38/R=0.27/F1=0.32 on the live holdout vs. P=0.80/R=0.64/F1=0.71 on CERTH, and the
6.3b visual audit ruled out an annotation-convention explanation -- this is a genuine recall failure driven
by a training-time density gap (CERTH averages 3.87 bunches/image, live photos average 10.4).

**This section is infrastructure, not yet a completed run.** Every cell below is gated behind
`FARM_FINETUNE_READY` (9.1), which requires enough labeled farm photos to make a real train/val/test split.
As of this writing that gate is **closed** (21-24 of 43 farm photos labeled) -- re-run 9.1 after finishing
Phase 3.1's annotation on the remaining photos, and the rest of Phase 9 will run for real. Nothing here
retrains the Phase 4 model or overwrites `domain_aware_v1_best.pt`; it produces a separate fine-tuned
checkpoint (9.6) so the original is never at risk.

**Important scoping note:** 9.1 splits *all* labeled farm photos (which will include the original 21-24
Phase 3.1 images) into a fresh farm_train/farm_val/farm_test. Some of those original images will likely end
up in farm_train here. That means Phase 5.4/6's `live_holdout` numbers are **not** a valid before/after
comparison for the fine-tuned model (some of its "before" test images become "after" training images) --
judge the fine-tuned model only against 9.5's `farm_test`, which is guaranteed disjoint from what it trained
on.

## 9.1 Split All Labeled Farm Photos (Gate)

Reads every label currently in `LIVE_HOLDOUT_LABELS` (Phase 3.1's ongoing annotation output -- this grows
as more photos get labeled, so re-running this cell after more annotation picks up the larger pool
automatically) and splits at the image level, 60/20/20, same seed as every other split in this notebook.

In [ ]:
FARM_LABEL_DIR = LIVE_HOLDOUT_LABELS
FARM_IMAGE_DIR = LIVE_HOLDOUT_IMAGES
FARM_FINETUNE_MIN_LABELED = 35   # floor for a workable 60/20/20 split (~21/7/7); labeling all 43 is better
FINETUNE_RUN_NAME = "farm_finetune_v1"   # defined here (not just inside 9.3) so 9.4-9.6 work even when 9.3
                                          # (the actual training) is skipped because it already ran previously

all_farm_labels = sorted(FARM_LABEL_DIR.glob("*.txt"))
n_farm_labeled = len(all_farm_labels)
FARM_FINETUNE_READY = n_farm_labeled >= FARM_FINETUNE_MIN_LABELED
print(f"Labeled farm photos: {n_farm_labeled} / {FARM_FINETUNE_MIN_LABELED} needed for Phase 9 to run.")

if not FARM_FINETUNE_READY:
    print("Not ready yet -- finish annotating in Phase 3.1, then re-run this cell (9.1) onward.")
    print("Every remaining Phase 9 cell checks FARM_FINETUNE_READY and no-ops until this passes.")
else:
    farm_stems = pd.DataFrame({"stem": [lf.stem for lf in all_farm_labels]})
    farm_train_df, farm_temp_df = train_test_split(farm_stems, test_size=0.4, random_state=SEED)
    farm_val_df, farm_test_df = train_test_split(farm_temp_df, test_size=0.5, random_state=SEED)
    print(f"Farm split -- train: {len(farm_train_df)}  val: {len(farm_val_df)}  test: {len(farm_test_df)}")
    print("NOTE: this split may include images also used in Phase 5.4/6's live_holdout eval -- see the")
    print("Phase 9 intro. Judge the fine-tuned model against farm_test (9.5), not live_holdout.")

## 9.2 Materialize the Fine-Tune Data Mix

Folds `farm_train` into the existing Zenodo+CERTH `train_mix` using the same share formula as Section 1.6,
applied a second time for a third domain layer. Two experimental levers are included, both **off by
default** -- read the caveats before enabling either.

- **`APPLY_CLAHE_TO_FARM_TRAIN`** (requires Phase 2.2 to have run this session -- it defines `apply_clahe()`) -- bakes CLAHE (2.2) into the farm-domain training images on disk. CLAHE
  was previously diagnostic-only (2.3's note above); this is the first place it would actually touch model
  input. If enabled, `farm_test` (9.5) must also be CLAHE-preprocessed before evaluation, or the comparison
  is invalid -- 9.5 handles this automatically by reading `APPLY_CLAHE_TO_FARM_TRAIN` itself.
- **`ENABLE_SYNTHETIC_DENSITY_AUG`** -- a custom compositor that pastes extra CERTH bunch crops onto CERTH
  base images to synthesize higher-density training scenes, at zero annotation cost. **A manual visual
  check of its output found visible paste seams** (lighting/exposure mismatch between donor crop and base
  image) on a meaningful fraction of composites -- a known failure mode of naive copy-paste without color
  matching, and a real risk of teaching the model "sharp rectangular color discontinuity = bunch" as a
  shortcut. Ultralytics' own built-in `copy_paste` (already in `AUGMENT_CONFIG`, bumped to 0.5 for this
  fine-tune below) does the same job with better-engineered blending and is the safer default lever for the
  same goal. Leave this off unless you've inspected `synth_dense_composite()`'s output yourself and are
  comfortable with the trade-off.

In [ ]:
APPLY_CLAHE_TO_FARM_TRAIN = False        # see 9.2 markdown -- experimental, must pair with 9.5's eval
ENABLE_SYNTHETIC_DENSITY_AUG = False     # see 9.2 markdown -- experimental, visible seam artifacts observed
TARGET_FARM_SHARE = 0.20                 # lower than CERTH's 0.5 (Section 1.6) -- far fewer farm images available
FINETUNE_LR0 = 0.002                     # well below the base run's 0.01 -- adapting a converged model, not training from scratch
FINETUNE_EPOCHS = 60
FINETUNE_PATIENCE = 15

FARM_FT_DATA = Path(ROOT, "data/processed_farm_ft")

def _clahe_copy(src_paths, dst_dir):
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in src_paths:
        img = cv2.imread(str(p))
        cv2.imwrite(str(dst_dir / p.name), apply_clahe(img))

def _plain_copy(src_paths, dst_dir):
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in src_paths:
        dst = dst_dir / p.name
        if not dst.exists():
            shutil.copy(p, dst)

def to_yolo_line(box, w, h, cls=0):
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
    bw, bh = (x2 - x1) / w, (y2 - y1) / h
    return f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"

def synth_dense_composite(base_path, base_label_path, donor_pool, n_paste=(2, 4), seed=None, max_tries=15):
    """Paste extra CERTH bunch crops onto a CERTH base image to synthesize a denser scene. See 9.2's
    caveat -- known to produce visible seams on some fraction of composites."""
    rng = random.Random(seed)
    base = Image.open(base_path).convert("RGB")
    w, h = base.size

    def read_boxes(lf, iw, ih):
        out = []
        if lf.exists():
            for line in open(lf):
                if not line.strip():
                    continue
                _, cx, cy, bw, bh = map(float, line.split())
                out.append([(cx - bw/2)*iw, (cy - bh/2)*ih, (cx + bw/2)*iw, (cy + bh/2)*ih])
        return out

    def box_iou(a, b):
        ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
        ix1, iy1 = max(ax1, bx1), max(ay1, by1); ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        inter = max(0., ix2-ix1) * max(0., iy2-iy1)
        ua = max(0.,ax2-ax1)*max(0.,ay2-ay1); ub = max(0.,bx2-bx1)*max(0.,by2-by1)
        u = ua+ub-inter
        return inter/u if u > 0 else 0.0

    boxes = read_boxes(base_label_path, w, h)
    for donor_img_path in rng.sample(donor_pool, min(rng.randint(*n_paste), len(donor_pool))):
        donor_label_path = Path(LOCAL_DATA, "train", "labels", f"{donor_img_path.stem}.txt")
        donor = Image.open(donor_img_path).convert("RGB")
        dw, dh = donor.size
        donor_boxes = read_boxes(donor_label_path, dw, dh)
        if not donor_boxes:
            continue
        bx1, by1, bx2, by2 = rng.choice(donor_boxes)
        pad_x, pad_y = (bx2-bx1)*0.15, (by2-by1)*0.15
        crop = donor.crop((int(max(0,bx1-pad_x)), int(max(0,by1-pad_y)), int(min(dw,bx2+pad_x)), int(min(dh,by2+pad_y))))
        scale = rng.uniform(0.8, 1.3)
        crop = crop.resize((max(1, int(crop.width*scale)), max(1, int(crop.height*scale))))
        cw, ch = crop.size
        if cw >= w or ch >= h or cw < 4 or ch < 4:
            continue
        for _ in range(max_tries):
            px, py = rng.randint(0, w - cw), rng.randint(0, h - ch)
            new_box = [px, py, px + cw, py + ch]
            if all(box_iou(new_box, b) < 0.15 for b in boxes):
                mask = Image.new("L", crop.size, 255)
                ImageDraw.Draw(mask).rectangle([0, 0, cw-1, ch-1], outline=0, width=1)
                mask = mask.filter(ImageFilter.GaussianBlur(8))
                base.paste(crop, (px, py), mask)
                boxes.append(new_box)
                break
    return base, boxes

if not FARM_FINETUNE_READY:
    print("Skipped -- FARM_FINETUNE_READY is False (see 9.1).")
else:
    from PIL import ImageFilter

    farm_train_paths = [FARM_IMAGE_DIR / f"{s}{next(FARM_IMAGE_DIR.glob(s+'.*')).suffix}" for s in farm_train_df["stem"]]
    copy_fn = _clahe_copy if APPLY_CLAHE_TO_FARM_TRAIN else _plain_copy
    copy_fn(farm_train_paths, FARM_FT_DATA / "farm_train" / "images")
    _plain_copy([FARM_LABEL_DIR / f"{s}.txt" for s in farm_train_df["stem"]], FARM_FT_DATA / "farm_train" / "labels")

    # Extend Section 1.6's mix formula with a third layer: fold farm_train into the existing train_mix.
    farm_pool_size = len(farm_train_paths)
    k_farm = 1
    combined_base = certh_pool_size * CERTH_OVERSAMPLE + S_o   # existing Zenodo+CERTH epoch size
    while True:
        farm_listings = k_farm * farm_pool_size
        share = farm_listings / (combined_base + farm_listings)
        if share >= TARGET_FARM_SHARE or k_farm >= 20:
            break
        k_farm += 1
    print(f"Farm oversample k={k_farm} -> {farm_pool_size * k_farm} effective listings "
          f"({(farm_pool_size * k_farm) / (combined_base + farm_pool_size * k_farm):.1%} of the fine-tune epoch)")

    farm_train_ft_paths = sorted((FARM_FT_DATA / "farm_train" / "images").glob("*.*")) * k_farm

    if ENABLE_SYNTHETIC_DENSITY_AUG:
        synth_dir_img = FARM_FT_DATA / "synth" / "images"
        synth_dir_lbl = FARM_FT_DATA / "synth" / "labels"
        synth_dir_img.mkdir(parents=True, exist_ok=True); synth_dir_lbl.mkdir(parents=True, exist_ok=True)
        certh_pool = [p for p in train_split["certh_images"] if Path(LOCAL_DATA, "train", "labels", f"{p.stem}.txt").exists()]
        n_synth = min(300, len(certh_pool))
        synth_paths = []
        for i, base_p in enumerate(random.Random(SEED).sample(certh_pool, n_synth)):
            base_lbl = Path(LOCAL_DATA, "train", "labels", f"{base_p.stem}.txt")
            img, boxes = synth_dense_composite(base_p, base_lbl, [p for p in certh_pool if p != base_p], seed=SEED + i)
            out_name = f"synth_{base_p.stem}_{i}.jpg"
            img.save(synth_dir_img / out_name, quality=90)
            w, h = img.size
            with open(synth_dir_lbl / f"synth_{base_p.stem}_{i}.txt", "w") as f:
                f.write("\n".join(to_yolo_line(b, w, h) for b in boxes))
            synth_paths.append(synth_dir_img / out_name)
        print(f"Synthetic density composites: {len(synth_paths)} images added to the fine-tune mix.")
    else:
        synth_paths = []

    finetune_train_mix = zenodo_subsample_images + certh_train_images * CERTH_OVERSAMPLE + farm_train_ft_paths + synth_paths
    random.Random(SEED).shuffle(finetune_train_mix)
    finetune_train_mix_path = FARM_FT_DATA / "train_mix_finetune.txt"
    FARM_FT_DATA.mkdir(parents=True, exist_ok=True)
    with open(finetune_train_mix_path, "w") as f:
        f.writelines(str(p.resolve()) + "\n" for p in finetune_train_mix)
    print(f"Fine-tune train mix: {len(finetune_train_mix)} listings -> {finetune_train_mix_path}")

    farm_val_paths = [FARM_IMAGE_DIR / f"{s}{next(FARM_IMAGE_DIR.glob(s+'.*')).suffix}" for s in farm_val_df["stem"]]
    copy_fn(farm_val_paths, FARM_FT_DATA / "farm_val" / "images")
    _plain_copy([FARM_LABEL_DIR / f"{s}.txt" for s in farm_val_df["stem"]], FARM_FT_DATA / "farm_val" / "labels")
    finetune_val_mix = online_val_subsample + certh_val_images + farm_val_paths
    finetune_val_mix_path = FARM_FT_DATA / "val_mix_finetune.txt"
    with open(finetune_val_mix_path, "w") as f:
        f.writelines(str(p.resolve()) + "\n" for p in finetune_val_mix)

    farm_test_paths = [FARM_IMAGE_DIR / f"{s}{next(FARM_IMAGE_DIR.glob(s+'.*')).suffix}" for s in farm_test_df["stem"]]
    copy_fn(farm_test_paths, FARM_FT_DATA / "farm_test" / "images")
    _plain_copy([FARM_LABEL_DIR / f"{s}.txt" for s in farm_test_df["stem"]], FARM_FT_DATA / "farm_test" / "labels")

    finetune_yaml = f"""path: {FARM_FT_DATA}
train: train_mix_finetune.txt
val: val_mix_finetune.txt
test: farm_test/images
nc: {NUM_CLASSES}
names: {CLASS_NAMES}
"""
    with open(FARM_FT_DATA / "data_finetune.yaml", "w") as f:
        f.write(finetune_yaml)
    print(finetune_yaml)

## 9.3 Fine-Tune From the Existing Checkpoint

Loads `domain_aware_v1_best.pt` directly (not `yolo11m.pt` from scratch) -- this is adapting an
already-converged detector to a new sub-domain with very few new images, not a cold start, so `lr0` is cut
well below the base run's 0.01 and the epoch budget is far shorter. `copy_paste` is bumped from the base
`AUGMENT_CONFIG`'s 0.3 to 0.5 (see 9.2) as the primary, library-backed density lever. Writes to a **new**
run name so `domain_aware_v1_best.pt` is never overwritten.

In [ ]:
if not FARM_FINETUNE_READY:
    print("Skipped -- FARM_FINETUNE_READY is False (see 9.1).")
else:
    FINETUNE_AUGMENT_CONFIG = {**AUGMENT_CONFIG, "copy_paste": 0.5}

    finetune_model = YOLO(str(SAVED_MODELS_DIR / "domain_aware_v1_best.pt"))
    finetune_results = finetune_model.train(
        data=str(FARM_FT_DATA / "data_finetune.yaml"),
        epochs=FINETUNE_EPOCHS,
        imgsz=TRAIN_IMGSZ,
        batch=-1,
        device=0,
        workers=8,
        seed=SEED,
        lr0=FINETUNE_LR0,
        project=RUNS_DIR,
        name=FINETUNE_RUN_NAME,
        patience=FINETUNE_PATIENCE,
        save_period=10,
        **FINETUNE_AUGMENT_CONFIG,
    )

## 9.4 Farm-Domain Memorization Check

Same pattern as Phase 4.4, applied to the fine-tune run specifically -- with only ~26 farm-train images
(before oversampling), overfitting risk here is real and worth checking directly rather than assuming
`copy_paste`/`mosaic` augmentation alone prevents it.

In [ ]:
if not FARM_FINETUNE_READY:
    print("Skipped -- FARM_FINETUNE_READY is False (see 9.1).")
else:
    ft_results_csv = Path(RUNS_DIR, FINETUNE_RUN_NAME, "results.csv")
    ft_df = pd.read_csv(ft_results_csv)
    ft_df.columns = [c.strip() for c in ft_df.columns]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ft_df["epoch"], ft_df["train/box_loss"], label="train box loss")
    ax.plot(ft_df["epoch"], ft_df["val/box_loss"], label="val (Zenodo/CERTH/farm mix) box loss")
    ax.set_xlabel("epoch"); ax.set_ylabel("box loss"); ax.legend()
    ax.set_title("Fine-tune: train vs. val loss -- divergence signals overfitting to the small farm-train slice")
    plt.show()

## 9.5 Evaluate the Fine-Tuned Model on `farm_test`

`farm_test` (9.1) is disjoint from everything the fine-tune trained/validated on -- this is the honest
number for whether fine-tuning actually helped. Also re-checks CERTH and Zenodo with the fine-tuned model to
confirm it didn't regress general bunch detection while adapting to the farm domain (the failure mode Phase
4.4-style checks exist to catch).

In [ ]:
if not FARM_FINETUNE_READY:
    print("Skipped -- FARM_FINETUNE_READY is False (see 9.1).")
else:
    ft_best_pt = Path(RUNS_DIR, FINETUNE_RUN_NAME, "weights", "best.pt")
    # Loaded fresh from disk rather than reusing 9.3's in-memory `finetune_model` -- this cell runs
    # correctly whether or not 9.3 executed in this kernel session (e.g. after skipping the already-
    # completed training, or resuming a fresh session), since the weights on disk are the source of truth.
    finetune_model = YOLO(str(ft_best_pt))
    ft_sahi_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics", model_path=str(ft_best_pt), confidence_threshold=0.25, device="cuda:0",
    )
    _orig_sahi_model = sahi_model
    sahi_model = ft_sahi_model   # evaluate_counts_sahi / evaluate_detection_metrics both close over this global

    farm_test_img_dir = FARM_FT_DATA / "farm_test" / "images"
    farm_test_lbl_dir = FARM_FT_DATA / "farm_test" / "labels"

    farm_test_conf_grid = np.concatenate([np.arange(0.01, 0.05, 0.01), np.arange(0.05, 0.61, 0.05)])
    ft_mae_by_conf = [evaluate_counts_sahi(farm_test_img_dir, farm_test_lbl_dir, conf=c, tag=f"farm_test conf={c:.2f}")["mae"]
                       for c in farm_test_conf_grid]
    ft_best_conf = farm_test_conf_grid[int(np.argmin(ft_mae_by_conf))]
    print(f"Fine-tuned model's own tuned confidence on farm_test: {ft_best_conf:.2f}")

    farm_test_count = evaluate_counts_sahi(farm_test_img_dir, farm_test_lbl_dir, conf=ft_best_conf, tag="farm_test -- fine-tuned model, tuned conf")
    farm_test_detection = evaluate_detection_metrics(farm_test_img_dir, farm_test_lbl_dir, conf=ft_best_conf, tag="farm_test -- fine-tuned model, tuned conf")

    print("\nRegression check -- fine-tuned model on the ORIGINAL domains (compare against Phase 5.2/5.3):")
    finetune_online_test = evaluate_counts(Path(LOCAL_DATA, "test", "images"), Path(LOCAL_DATA, "test", "labels"),
                                            finetune_model, tag="Online/Zenodo test -- fine-tuned model", label_files=test_split["zenodo_labels"])
    finetune_certh_holdout = evaluate_counts_sahi(Path(LOCAL_DATA, "test", "images"), Path(LOCAL_DATA, "test", "labels"),
                                                   conf=0.25, tag="CERTH holdout -- fine-tuned model", label_files=test_split["certh_labels"])
    print(f"\nCompare to pre-fine-tune baseline: online_test MAE={online_test['mae']:.2f}, "
          f"certh_holdout_sahi MAE={certh_holdout_sahi['mae']:.2f} (Phase 5.2/5.6). A fine-tune that wins on")
    print("farm_test but loses meaningfully on these is not an unambiguous improvement -- weigh both.")

    sahi_model = _orig_sahi_model   # restore -- downstream cells (if re-run) should use the original model again

## 9.6 Save the Fine-Tuned Model

Mirrors Phase 4.3 -- saved under its own name so `domain_aware_v1_best.pt` (the model every earlier phase's
numbers describe) is untouched.

In [ ]:
import json as _json

if not FARM_FINETUNE_READY:
    print("Skipped -- FARM_FINETUNE_READY is False (see 9.1).")
else:
    for _fname in ["best.pt", "last.pt"]:
        _src = Path(RUNS_DIR, FINETUNE_RUN_NAME, "weights", _fname)
        if _src.exists():
            shutil.copy(_src, SAVED_MODELS_DIR / f"{FINETUNE_RUN_NAME}_{_fname}")

    _ft_meta = {
        "run_name": FINETUNE_RUN_NAME,
        "base_checkpoint": "domain_aware_v1_best.pt",
        "farm_train_images": len(farm_train_df), "farm_val_images": len(farm_val_df), "farm_test_images": len(farm_test_df),
        "target_farm_share": TARGET_FARM_SHARE, "farm_oversample_k": k_farm,
        "apply_clahe_to_farm_train": APPLY_CLAHE_TO_FARM_TRAIN,
        "enable_synthetic_density_aug": ENABLE_SYNTHETIC_DENSITY_AUG,
        "lr0": FINETUNE_LR0, "epochs": FINETUNE_EPOCHS,
        "farm_test_mae": farm_test_count["mae"], "farm_test_f1": farm_test_detection["f1"],
        "saved_at": datetime.datetime.now().isoformat(timespec="seconds")
    }
    with open(SAVED_MODELS_DIR / f"{FINETUNE_RUN_NAME}_meta.json", "w") as f:
        _json.dump(_ft_meta, f, indent=2)
    print(f"Saved fine-tuned weights + metadata to {SAVED_MODELS_DIR}/ as {FINETUNE_RUN_NAME}_*")